# DPC through the LayerNorm surrogates: does the kink hurt control, and does the slope loss fix it?

**Question.** A controller trained by Differentiable Predictive Control (DPC) learns from the model's gradients. Does the
LayerNorm kink make that controller worse on the TRUE plant, and does Level 1 training (the Jacobian / slope loss) bring
it back to the level of the kink-free models?

**Pipeline** (your DPC example, applied to this study):
1. *Identify the plant*: retrain the Level 1 surrogates exactly as in the Level 1 notebook (same data, initial weights,
   labels and slope-check draws; the 3000-step models match the Level 1 run to every printed digit). Plant
   `x+ = 1.03 x + 0.01 u`, 20 000 noisy transitions, x in [-4, 4], u in [-10, 10].
2. *Train a GRU controller by unrolling it through the learned model* `x_{k+1} = F(x_k, u_k)` - the whole network,
   so the controller sees the kink. Loss = your LQR-type loss (Q_x = 5, R_u = 0.1, Q_f = 50, divided by N), horizon
   curriculum 5 -> 10 -> 20 -> 50 -> 100 -> 200 (1000 / 2000 / 2000 / 1000 / 1000 / 2000 epochs), AdamW 1e-3,
   1000 initial states x0 ~ U[-3, 3], full batch.
3. *Test on the TRUE plant.* Reference = the same controller trained through the true plant ("oracle"). Headline =
   the true-plant cost gap.

**Changes from the example, each needed here:** (a) rollout through the whole surrogate, not W and V after LayerNorm
(the example's own warning: that is not the model, and the kink would never be seen); (b) `u = 10 tanh(head)`: the
surrogates only saw |u| <= 10 and the actuator is limited there (unbounded u would test extrapolation, not the kink);
(c) plain JAX instead of flax.nnx so the 10 seeds train in parallel (same GRU gates, `optax.adamw`); (d) per seed,
every model's controller starts from the same weights and trains on the same 1000 initial states: paired comparisons.

**Models** (one controller per model per seed, 10 seeds): true plant (oracle); LayerNorm MSE 3000 steps (the problem);
LayerNorm MSE 12 000 steps ("just train longer"); LayerNorm J_x + J_u from step 0; LayerNorm two-stage (3000 MSE, then
1000 MSE + slope loss); no norm and static norm (kink-free yardsticks).

**Task: regulation to x = 0.** Every trajectory ends at (x, u) = (0, 0), within about 0.1 of every model's kink centre:
the most kink-exposed task for this plant.

**Predictions, written before the run (report the outcome either way):**
* **P1** LayerNorm MSE does worse than the kink-free models, mainly from small starts (which begin inside the kink);
  about the same from large starts.
* **P2** Two-stage falls within the kink-free range.
* **P4** MSE 12 000 lands in between, failing mainly on the narrow-kink seeds (1, 2, 8, 9).
(P3 of the earlier plan concerned sampling-based MPC, which is not run here.)

**Extension (2026-10-22): repeats and a diagnostic.** (1) `REPS = [0, 1, 2]`: every controller is trained three times,
each repeat with a fresh controller initialisation and a fresh draw of its 1000 training starts. Repeat 0 is exactly the
first full run, so **point the notebook at the same output folder** (on Colab: `MyDrive/dpc_out`, the default) and only
repeats 1 and 2 train (about 1-1.5 h on an A100); the surrogates and labels are reused too. Question: is a failure a
property of the surrogate (it fails in every repeat) or of one unlucky DPC training? (2) A diagnostic that needs no
training: the controller trained through the TRUE plant is run through each surrogate. If the surrogate rates that good
controller about as the plant does, but rates the controller DPC found through it much worse, a good controller existed
for the surrogate and DPC missed it: an optimisation trap caused by the wrong slopes DPC learns from, not a model whose
predictions mislead.

**Extension (2026-10-23): controller architecture.** `ARCH` selects the controller: `"gru"` (the original study, unchanged),
`"mlp"` (memoryless u = 10 tanh(MLP(x)), 2 x 64 tanh) or `"linear"` (u ~ sat(K x + c), 2 parameters). Non-GRU results go to
`dpc_out_mlp` / `dpc_out_linear` and reuse the surrogates in `dpc_out`. See `NOTES_controller_arch.md` (predictions A1-A7).

**Extension (2026-10-23): robust per-start gradients** (last section before "Save everything"): the optimiser-side control
suggested by the trap mechanism. With the full run already in the output folder, only the two robust models train.

**Where to run.** The notebook works on Colab and Kaggle. On Colab it mounts Google Drive and writes to `MyDrive/dpc_out`, so
a disconnect loses nothing: re-run and it continues from the last finished model.

| GPU | Setting | Why |
|---|---|---|
| **A100** (Colab) - recommended | `DTYPE = "float64"` | fast 64-bit arithmetic: same precision as Level 1 |
| L4, G4 (Colab), T4 (Kaggle/Colab) | `DTYPE = "float32"` | these GPUs are slow in 64-bit; 32-bit runs at full precision (see below) |

The surrogates are always trained in float64 (as in Level 1); `DTYPE` only sets the precision of the controller training and
the closed-loop tests. With float32 the notebook forces full-precision matrix products (`jax_default_matmul_precision =
"highest"`): by default, recent NVIDIA GPUs multiply float32 matrices in TF32 (about 3 significant digits), too coarse near the
kink, where LayerNorm divides by about 0.03.

Run once with `QUICK = True` (5-10 min) to check the GPU and the time per model, then `QUICK = False` and run all. Measured
cost: ~6.7 s per epoch at horizon 200 on a 2-core CPU (float64), i.e. ~6 h per model on CPU; a GPU should take 1-3 h for all
7 models (the quick run's per-model times tell you).

In [ ]:
import importlib.util, subprocess, sys
for pkg in ("jax", "optax"):
    if importlib.util.find_spec(pkg) is None:      # Kaggle images usually ship both; otherwise needs Internet ON
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import jax
print("jax", jax.__version__, "| devices:", jax.devices())

In [ ]:
%%writefile level1_core.py
"""level1_core.py - Level 1 of the LayerNorm-kink method.

Level 1 = MSE + lambda x (Jacobian loss with labels estimated from the same transitions), the
slope-check points drawn uniformly from the training inputs. No lens, no trust weights, no other
tricks beyond the schedules being compared.

Everything is identical to smc_core.py (same plant, data seeds, model, initialisation, optimiser,
label estimator, k selection, sampling of slope-check points), so the numbers are comparable with
the E1 runs. The only differences: no label-trust weights (Level 1 has none; on box data they are
all 1 anyway), the Jacobian loss can use J_x only, J_u only or both, and the closed-form kink-line
evaluation points are clipped into the box (fixed count, so nothing recompiles).
"""
import os
import csv
import time
import numpy as np
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from scipy.spatial import cKDTree

tmap = jax.tree_util.tree_map

# ------------------------------------------------------------------------------------------------ #
# 0. Problem setup (Level 0, inherited from the TMLR theory paper)                                 #
# ------------------------------------------------------------------------------------------------ #
XMAX, UMAX = 4.0, 10.0          # data box: x ~ U[-4, 4], u ~ U[-10, 10]
N_DATA, N_TEST = 20000, 4000    # training transitions, noiseless test points
H, WB, EPS = 16, 32, 1e-6       # encoder width, MLP width, LayerNorm epsilon
LR = 3e-3                       # Adam learning rate, full batch
LQR_Q, LQR_R = 5.0, 0.1         # LQR weights of the closed-loop screen
K_GRID = (40, 80, 160, 320, 640, 1280)


class Plant:
    """x+ = A x + B u + eps_nl (a_sin sin x + c_xu x u). eps_nl = 0 is the linear plant."""

    def __init__(self, A=1.03, B=0.01, eps_nl=0.0, a_sin=0.05, c_xu=0.002):
        self.A, self.B, self.eps = float(A), float(B), float(eps_nl)
        self.a_sin, self.c_xu = float(a_sin), float(c_xu)

    def f(self, x, u):
        return self.A * x + self.B * u + self.eps * (self.a_sin * np.sin(x) + self.c_xu * x * u)

    def Jx(self, x, u):
        return self.A + self.eps * (self.a_sin * np.cos(x) + self.c_xu * u)

    def Ju(self, x, u):
        return self.B + self.eps * self.c_xu * x + 0.0 * u

    def u_ss(self, r):
        r = np.asarray(r, float)
        den = self.B + self.eps * self.c_xu * r
        with np.errstate(divide="ignore", invalid="ignore"):
            return (r - self.A * r - self.eps * self.a_sin * np.sin(r)) / den

    def r_max(self):
        """Setpoints the actuator can hold with a 10 % margin."""
        return min(0.9 * self.B * UMAX / max(self.A - 1.0, 1e-6), XMAX)


def make_data(plant, sigma, data_seed, n=N_DATA):
    rng = np.random.default_rng(data_seed)
    x = rng.uniform(-XMAX, XMAX, n)
    u = rng.uniform(-UMAX, UMAX, n)
    y = plant.f(x, u) + sigma * rng.standard_normal(n)
    return dict(x=x, u=u, y=y)


def make_test(plant):
    return make_data(plant, 0.0, 12345, N_TEST)


# ------------------------------------------------------------------------------------------------ #
# 1. Jacobian labels from the transitions (local quadratic, k nearest in std-scaled distance)      #
# ------------------------------------------------------------------------------------------------ #
def kernel_weights(t, kernel="epan"):
    """t = d/R in [0,1]. 'epan': 1 - t^2 (current code). 'gauss': exp(-2 t^2) (the note's rule)."""
    t = np.clip(t, 0.0, 1.0)
    if kernel == "epan":
        w = 1.0 - t ** 2
    elif kernel == "gauss":
        w = np.exp(-2.0 * t ** 2)
    else:
        raise ValueError(kernel)
    return np.maximum(w, 1e-12)


def _design(tx, tu):
    one = np.ones_like(tx)
    return np.stack([one, tx, tu, 0.5 * tx ** 2, tx * tu, 0.5 * tu ** 2], -1)


class LabelEstimator:
    """Local-quadratic weighted least squares around each query (x0, u0).
    d_i = sqrt(((x_i-x0)/s_x)^2 + ((u_i-u0)/s_u)^2), k nearest, R = d_(k), w_i = kernel(d_i/R),
    t = offsets / (s R), J_x = beta_1/(s_x R), J_u = beta_2/(s_u R). s_x, s_u = data std.
    Also returns v = Var(label)/sigma^2 from the sandwich formula."""

    def __init__(self, x, u, y, k, kernel="epan", ridge=1e-8):
        self.x, self.u, self.y = (np.asarray(a, float) for a in (x, u, y))
        self.k, self.kernel, self.ridge = int(k), kernel, float(ridge)
        self.sx, self.su = float(self.x.std()), float(self.u.std())
        self.tree = cKDTree(np.column_stack([self.x / self.sx, self.u / self.su]))

    def __call__(self, xq, uq, want_var=True):
        xq, uq = np.atleast_1d(np.asarray(xq, float)), np.atleast_1d(np.asarray(uq, float))
        chunk = max(32, 400_000 // self.k)
        n = len(xq)
        out = {k: np.empty(n) for k in ("Jx", "Ju", "vx", "vu", "R")}
        P = np.eye(6)
        P[0, 0] = 0.0
        for s in range(0, n, chunk):
            xs, us = xq[s:s + chunk], uq[s:s + chunk]
            d, idx = self.tree.query(np.column_stack([xs / self.sx, us / self.su]), k=self.k)
            R = np.maximum(d[:, -1:], 1e-12)
            w = kernel_weights(d / R, self.kernel)
            tx = (self.x[idx] - xs[:, None]) / (self.sx * R)
            tu = (self.u[idx] - us[:, None]) / (self.su * R)
            Phi = _design(tx, tu)
            PhiW = Phi * w[..., None]
            G = np.einsum("mki,mkj->mij", PhiW, Phi) + self.ridge * P
            Ginv = np.linalg.inv(G)
            beta = np.einsum("mij,mj->mi", Ginv, np.einsum("mki,mk->mi", PhiW, self.y[idx]))
            Rs = R[:, 0]
            out["Jx"][s:s + chunk] = beta[:, 1] / (self.sx * Rs)
            out["Ju"][s:s + chunk] = beta[:, 2] / (self.su * Rs)
            out["R"][s:s + chunk] = Rs
            if want_var:
                Cov = Ginv @ np.einsum("mki,mkj->mij", PhiW * w[..., None], Phi) @ Ginv
                out["vx"][s:s + chunk] = Cov[:, 1, 1] / (self.sx * Rs) ** 2
                out["vu"][s:s + chunk] = Cov[:, 2, 2] / (self.su * Rs) ** 2
        return out

    def noise_var(self, n_probe=2000, seed=0):
        """Pooled residual variance of unweighted local quadratic fits: an estimate of sigma^2."""
        rng = np.random.default_rng(seed)
        sel = rng.choice(len(self.x), size=min(n_probe, len(self.x)), replace=False)
        d, idx = self.tree.query(np.column_stack([self.x[sel] / self.sx, self.u[sel] / self.su]), k=self.k)
        R = np.maximum(d[:, -1:], 1e-12)
        Phi = _design((self.x[idx] - self.x[sel, None]) / (self.sx * R), (self.u[idx] - self.u[sel, None]) / (self.su * R))
        Yn = self.y[idx]
        G = np.einsum("mki,mkj->mij", Phi, Phi) + 1e-12 * np.eye(6)
        beta = np.linalg.solve(G, np.einsum("mki,mk->mi", Phi, Yn)[..., None])[..., 0]
        res = Yn - np.einsum("mki,mi->mk", Phi, beta)
        return float((res ** 2).sum() / (res.size - 6 * len(sel)))


def select_k(x, u, y, ks=K_GRID, kernel="epan", n_query=2000, seed=0):
    """k minimising the estimated label MSE (bias^2 + variance) in loss units, with no true Jacobian:
    variance = sigma_hat^2 x sandwich factor; bias^2(k) ~ mean[(J(2k) - J(k))^2] - sigma_hat^2 (v_k - v_2k)
    (doubling trick: the slope bias grows ~ linearly in k in 2-D); score = Var(x) MSE_x + Var(u) MSE_u.
    Returns (k, table, sigma_hat^2)."""
    x, u, y = (np.asarray(a, float) for a in (x, u, y))
    rng = np.random.default_rng(seed)
    q = rng.choice(len(x), min(n_query, len(x)), replace=False)
    xq, uq = x[q], u[q]
    ks = sorted(set(int(k) for k in ks))
    allk = [k for k in sorted(set(ks + [2 * k for k in ks])) if k < len(x)]
    labs = {k: LabelEstimator(x, u, y, k, kernel)(xq, uq) for k in allk}
    s2 = LabelEstimator(x, u, y, ks[0], kernel).noise_var()
    rows = []
    for k in ks:
        if 2 * k not in labs:
            continue
        a, b = labs[k], labs[2 * k]
        row = dict(k=k)
        for J, v in (("Jx", "vx"), ("Ju", "vu")):
            var_k = s2 * np.mean(a[v])
            bias2 = max(np.mean((b[J] - a[J]) ** 2) - s2 * (np.mean(a[v]) - np.mean(b[v])), 0.0)
            row[f"bias2_{J}"], row[f"var_{J}"], row[f"mse_{J}"] = bias2, var_k, bias2 + var_k
        row["score"] = np.var(x) * row["mse_Jx"] + np.var(u) * row["mse_Ju"]
        rows.append(row)
    best = min(rows, key=lambda r: r["score"])["k"]
    return best, rows, s2


def get_labels(d, seed, sigma, kernel, cache_dir):
    """Labels at every training input, computed once per dataset and cached to disk."""
    fn = os.path.join(cache_dir, f"labels_seed{seed}_sigma{sigma:g}_{kernel}.npz")
    if os.path.exists(fn):
        z = np.load(fn)
        return {k: z[k] for k in z.files}
    t0 = time.time()
    k, table, s2 = select_k(d["x"], d["u"], d["y"], kernel=kernel)
    lab = LabelEstimator(d["x"], d["u"], d["y"], k, kernel)(d["x"], d["u"])
    out = dict(Jx=lab["Jx"], Ju=lab["Ju"], vx=lab["vx"], vu=lab["vu"], k=np.array(k), sigma_hat=np.array(np.sqrt(s2)),
               ktab_k=np.array([r["k"] for r in table]), ktab_score=np.array([r["score"] for r in table]),
               ktab_bias2_Ju=np.array([r["bias2_Ju"] for r in table]), ktab_var_Ju=np.array([r["var_Ju"] for r in table]),
               ktab_bias2_Jx=np.array([r["bias2_Jx"] for r in table]), ktab_var_Jx=np.array([r["var_Jx"] for r in table]),
               time_s=np.array(time.time() - t0))
    np.savez_compressed(fn, **out)
    return out


# ------------------------------------------------------------------------------------------------ #
# 2. The surrogate (identical to kink_core / smc_core)                                             #
# ------------------------------------------------------------------------------------------------ #
def _glorot(key, shape):
    lim = np.sqrt(6.0 / (shape[0] + shape[1]))
    return jax.random.uniform(key, shape, minval=-lim, maxval=lim, dtype=jnp.float64)


def init_params(seed):
    k = jax.random.split(jax.random.PRNGKey(int(seed)), 4)
    kb = jax.random.split(k[2], 8)
    return dict(E=_glorot(k[0], (2, H)), b=jnp.zeros(H), gamma=jnp.ones(H), beta=jnp.zeros(H),
                Wd=_glorot(k[1], (H, 1)), bd=jnp.zeros(1),
                br=dict(W1=_glorot(kb[0], (H, WB)), b1=jnp.zeros(WB), W2=_glorot(kb[1], (WB, H)), b2=jnp.zeros(H)))


def stack_trees(trees):
    return tmap(lambda *a: jnp.stack(a), *trees)


def unstack_tree(tree, i):
    return tmap(lambda a: a[i], tree)


def _norm(kind, h, s0):
    if kind == "none":
        return h
    r = h - jnp.mean(h, -1, keepdims=True)
    if kind == "layer":
        return r / jnp.sqrt(jnp.mean(r ** 2, -1, keepdims=True) + EPS)
    if kind == "static":
        return r / s0
    raise ValueError(kind)


def F_point(p, x, u, s0, norm):
    h = x * p["E"][0] + u * p["E"][1] + p["b"]
    v = _norm(norm, h, s0)
    a = jax.nn.gelu((p["gamma"] * v + p["beta"]) @ p["br"]["W1"] + p["br"]["b1"], approximate=False)
    return h @ p["Wd"][:, 0] + (a @ p["br"]["W2"] + p["br"]["b2"]) @ p["Wd"][:, 0] + p["bd"][0]


def frozen_sigma(p, x, u):
    """Scale of the 'static' normalisation: the LayerNorm scale at initialisation, averaged over the data."""
    h = np.asarray(x)[:, None] * np.asarray(p["E"][0]) + np.asarray(u)[:, None] * np.asarray(p["E"][1]) + np.asarray(p["b"])
    r = h - h.mean(-1, keepdims=True)
    return float(np.sqrt((r ** 2).mean(-1) + EPS).mean())


def _Fb(norm):
    return jax.vmap(lambda p, x, u, s0: F_point(p, x, u, s0, norm), in_axes=(None, 0, 0, None))


def _derivs(Fb, p, x, u, s0):
    jx = jax.jvp(lambda xx: Fb(p, xx, u, s0), (x,), (jnp.ones_like(x),))[1]
    ju = jax.jvp(lambda uu: Fb(p, x, uu, s0), (u,), (jnp.ones_like(u),))[1]
    return jx, ju


# ------------------------------------------------------------------------------------------------ #
# 3. Loss and trainer (vmapped over seeds)                                                         #
# ------------------------------------------------------------------------------------------------ #
def make_loss(norm, kind, M):
    """kind 'mse':  mean (F - y)^2
       kind 'sob':  mean (F - y)^2 + lam * mean_j [ wx s_x^2 (dF/dx - Jx_lab)^2 + wu s_u^2 (dF/du - Ju_lab)^2 ]
    over M slope-check points drawn uniformly (with replacement) from the training inputs each step.
    wx, wu in {0, 1} select J_x only / J_u only / both. s_x, s_u = data std (unit-invariant loss)."""
    Fb = _Fb(norm)

    def loss(p, key, D, wx, wu, lam):
        mse = jnp.mean((Fb(p, D["x"], D["u"], D["s0"]) - D["y"]) ** 2)
        if kind == "mse":
            return mse
        n = D["x"].shape[0]
        idx = jnp.clip(jnp.searchsorted(D["cdf"], jax.random.uniform(key, (M,), dtype=jnp.float64)), 0, n - 1)
        jx, ju = _derivs(Fb, p, D["x"][idx], D["u"][idx], D["s0"])
        reg = jnp.mean(wx * D["sx"] ** 2 * (jx - D["Jx_lab"][idx]) ** 2 + wu * D["su"] ** 2 * (ju - D["Ju_lab"][idx]) ** 2)
        return mse + lam * reg

    return loss


_TRAINERS = {}


def get_trainer(norm, kind, M, chunk):
    key = (norm, kind, M, chunk)
    if key in _TRAINERS:
        return _TRAINERS[key]
    loss = make_loss(norm, kind, M)
    b1, b2, ae = 0.9, 0.999, 1e-8

    def one_chunk(p, m, v, t0, key, D, wx, wu, lam):
        def st(c, i):
            p, m, v, key = c
            key, sub = jax.random.split(key)
            t = t0 + i + 1.0
            g = jax.grad(loss)(p, sub, D, wx, wu, lam)
            m = tmap(lambda a, b: b1 * a + (1 - b1) * b, m, g)
            v = tmap(lambda a, b: b2 * a + (1 - b2) * b * b, v, g)
            p = tmap(lambda a, mm, vv: a - LR * (mm / (1 - b1 ** t)) / (jnp.sqrt(vv / (1 - b2 ** t)) + ae), p, m, v)
            return (p, m, v, key), 0.0
        (p, m, v, key), _ = jax.lax.scan(st, (p, m, v, key), jnp.arange(chunk, dtype=jnp.float64))
        return p, m, v, key

    run = jax.jit(jax.vmap(one_chunk, in_axes=(0, 0, 0, None, 0, 0, None, None, None)))
    _TRAINERS[key] = run
    return run


_EVALS = {}


def get_evaluator(norm):
    """Batched over seeds: (P, X[S,n], U[S,n], s0[S]) -> F, dF/dx, dF/du, each [S, n]."""
    if norm in _EVALS:
        return _EVALS[norm]
    Fb = _Fb(norm)

    def one(p, x, u, s0):
        jx, ju = _derivs(Fb, p, x, u, s0)
        return Fb(p, x, u, s0), jx, ju

    ev = jax.jit(jax.vmap(one, in_axes=(0, 0, 0, 0)))
    _EVALS[norm] = ev
    return ev


# ------------------------------------------------------------------------------------------------ #
# 4. Kink geometry (closed form, from the encoder weights) and evaluation                          #
# ------------------------------------------------------------------------------------------------ #
_PM = np.eye(H) - np.ones((H, H)) / H


def kink_center(p):
    """z_min = argmin ||P(E^T z + b)|| and the kink half-widths r*_x, r*_u (Theorem U2 of the theory paper)."""
    E, b = np.asarray(p["E"], float), np.asarray(p["b"], float)
    A = _PM @ E.T
    c0 = _PM @ b
    z = -np.linalg.lstsq(A, c0, rcond=None)[0]
    r = A @ z + c0
    D = np.sqrt(r @ r + H * EPS)
    return z, D / np.linalg.norm(A[:, 0]), D / np.linalg.norm(A[:, 1])


def kink_line_points(p, n_lines=41, n_pts=61, tmax=8.0):
    """Points on the closed-form kink lines (x-sweeps at fixed u and u-sweeps at fixed x), each dense around
    its own closest-approach point. Clipped into the box so the count is fixed (2 n_lines n_pts)."""
    E, b = np.asarray(p["E"], float), np.asarray(p["b"], float)
    th = np.tan(np.linspace(-np.arctan(tmax), np.arctan(tmax), n_pts))
    out = []
    for sweep, fixed, lim in ((0, np.linspace(-UMAX, UMAX, n_lines), XMAX), (1, np.linspace(-XMAX, XMAX, n_lines), UMAX)):
        q = _PM @ E[sweep]
        nq2 = float(q @ q)
        C = fixed[:, None] * (_PM @ E[1 - sweep])[None, :] + (_PM @ b)[None, :]
        s_star = -(C @ q) / nq2
        cp = C + s_star[:, None] * q[None, :]
        rstar = np.sqrt((cp ** 2).sum(1) + H * EPS) / np.sqrt(nq2)
        s = np.clip(s_star[:, None] + rstar[:, None] * th[None, :], -lim, lim)
        f0 = np.repeat(fixed[:, None], n_pts, 1)
        xy = np.stack([s, f0], -1) if sweep == 0 else np.stack([f0, s], -1)
        out.append(xy.reshape(-1, 2))
    return np.concatenate(out, 0)


def grid(n):
    xg, ug = np.meshgrid(np.linspace(-XMAX, XMAX, n), np.linspace(-UMAX, UMAX, n))
    return xg.ravel(), ug.ravel()


def dare_K(a, b, Q=LQR_Q, R=LQR_R):
    a, b = np.asarray(a, float), np.asarray(b, float)
    b2 = b * b
    coef = R - Q * b2 - a * a * R
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.where(b2 > 1e-14, (-coef + np.sqrt(coef ** 2 + 4 * b2 * Q * R)) / (2 * b2), np.nan)
        return np.where(b2 > 1e-14, a * b * P / (R + b2 * P), 0.0)


def eq_line(plant, n=801):
    rm = plant.r_max()
    r = np.linspace(-rm, rm, n)
    us = plant.u_ss(r)
    ok = np.isfinite(us) & (np.abs(us) <= UMAX)
    return r[ok], us[ok]


def lqr_screen(a_h, b_h, plant, r, us):
    """LQR (Q, R) designed from the MODEL's local (A^, B^) at each setpoint on the equilibrium line, applied
    to the TRUE local plant. 'local' uses the model Jacobian at the setpoint; 'win' averages it over +-W."""
    a_t, b_t = plant.Jx(r, us), plant.Ju(r, us)
    pole = a_t - b_t * dare_K(a_h, b_h)
    pole_o = a_t - b_t * dare_K(a_t, b_t)
    W = 0.25 * plant.B * UMAX / max(plant.A - 1.0, 1e-6) / 3.33
    k = max(1, int(round(W / max(r[1] - r[0], 1e-12))))
    ker = np.ones(2 * k + 1)
    nrm = np.convolve(np.ones_like(a_h), ker, "same")
    a_w, b_w = np.convolve(a_h, ker, "same") / nrm, np.convolve(b_h, ker, "same") / nrm
    pole_w = a_t - b_t * dare_K(a_w, b_w)
    return dict(lqr_unstable=float(np.mean(np.abs(pole) >= 1.0)), lqr_unstable_win=float(np.mean(np.abs(pole_w) >= 1.0)),
                lqr_pole_err_max=float(np.max(np.abs(pole - pole_o))), eq_gain_min=float(np.min(b_h / b_t)),
                eq_gain_max=float(np.max(b_h / b_t)))


CENTRE_WIN = (0.5, 1.5)    # |x| <= 0.5, |u| <= 1.5: 1.9 % of the box around (0, 0), where the LayerNorm kink sits


def kink_tests(E, X, U, sx=XMAX / np.sqrt(3.0), su=UMAX / np.sqrt(3.0)):
    """Four measurable signatures that separate a localised kink from smooth approximation error, computed from the
    input-gain error E = (dF/du - B)/B on the evaluation grid (E: [S, n, n], rows = u, columns = x)."""
    E = np.asarray(E, float)
    A = np.abs(E)
    S = E.shape[0]
    rms = np.sqrt((E ** 2).reshape(S, -1).mean(1))
    peak = A.reshape(S, -1).max(1)
    win = (np.abs(X) <= CENTRE_WIN[0]) & (np.abs(U) <= CENTRE_WIN[1])
    centre = (E ** 2)[:, win].sum(1) / np.maximum((E ** 2).reshape(S, -1).sum(1), 1e-300)
    dx, du = X[0, 1] - X[0, 0], U[1, 0] - U[0, 0]
    gU, gX = np.gradient(E, du, dx, axis=(1, 2))
    steep = np.sqrt((gX * sx) ** 2 + (gU * su) ** 2).reshape(S, -1).max(1)
    k = A.reshape(S, -1).argmax(1)
    wx, wu = X.ravel()[k], U.ravel()[k]
    xmax, umax = np.abs(X).max(), np.abs(U).max()
    where = np.where((np.abs(wx) <= 2 * CENTRE_WIN[0]) & (np.abs(wu) <= 2 * CENTRE_WIN[1]), "centre",
                     np.where((np.abs(wx) >= 0.95 * xmax) | (np.abs(wu) >= 0.95 * umax), "edge", "interior"))
    return dict(crest=peak / np.maximum(rms, 1e-300), centre_energy=centre, steep=steep, worst_x=wx, worst_u=wu, worst_where=where)


def evaluate_batch(P, S0, norm, plant, test, ngrid, full=False):
    """Metrics for every seed of a stacked model P. Never touches training data."""
    ev = get_evaluator(norm)
    S = len(S0)
    s0 = jnp.asarray(S0)
    tile = lambda a: jnp.asarray(np.repeat(np.asarray(a)[None], S, 0))
    Ft, _, _ = ev(P, tile(test["x"]), tile(test["u"]), s0)
    tmse = np.mean((np.asarray(Ft) - test["y"][None]) ** 2, 1)
    xg, ug = grid(ngrid)
    _, jxg, jug = ev(P, tile(xg), tile(ug), s0)
    jxg, jug = np.asarray(jxg), np.asarray(jug)
    ex, eu = jxg - plant.Jx(xg, ug)[None], (jug - plant.Ju(xg, ug)[None]) / plant.B
    out = dict(rel_mse=tmse / np.var(test["y"]), Jx_rms=np.sqrt((ex ** 2).mean(1)), Ju_rms_rel=np.sqrt((eu ** 2).mean(1)),
               Jx_sup=np.abs(ex).max(1), Ju_sup_rel=np.abs(eu).max(1),
               flip_area=np.mean(np.sign(jug) != np.sign(plant.Ju(xg, ug))[None], 1))
    inner = (np.abs(xg) <= 0.95 * XMAX) & (np.abs(ug) <= 0.95 * UMAX)        # box without its outer 5 % (edge effects)
    out["flip_area_interior"] = np.mean((np.sign(jug) != np.sign(plant.Ju(xg, ug))[None])[:, inner], 1)
    if norm == "layer":
        KL = np.stack([kink_line_points(unstack_tree(P, i)) for i in range(S)])
        _, jxk, juk = ev(P, jnp.asarray(KL[..., 0]), jnp.asarray(KL[..., 1]), s0)
        ekx = np.abs(np.asarray(jxk) - plant.Jx(KL[..., 0], KL[..., 1])).max(1)
        eku = np.abs((np.asarray(juk) - plant.Ju(KL[..., 0], KL[..., 1])) / plant.B).max(1)
        out["Jx_sup"] = np.maximum(out["Jx_sup"], ekx)
        out["Ju_sup_rel"] = np.maximum(out["Ju_sup_rel"], eku)
    if not full:
        return out, None
    out["gain_bad_area"] = np.mean(np.abs(jug / plant.Ju(xg, ug)[None] - 1.0) > 1.0, 1)
    out.update(kink_tests(eu.reshape(S, ngrid, ngrid), xg.reshape(ngrid, ngrid), ug.reshape(ngrid, ngrid)))
    r, us = eq_line(plant)
    _, jxe, jue = ev(P, tile(r), tile(us), s0)
    jxe, jue = np.asarray(jxe), np.asarray(jue)
    scr = [lqr_screen(jxe[i], jue[i], plant, r, us) for i in range(S)]
    for key in scr[0]:
        out[key] = np.array([s[key] for s in scr])
    geo = [kink_center(unstack_tree(P, i)) for i in range(S)] if norm == "layer" else [(np.full(2, np.nan), np.nan, np.nan)] * S
    out["zmin_x"] = np.array([g[0][0] for g in geo])
    out["zmin_u"] = np.array([g[0][1] for g in geo])
    out["rstar_x"] = np.array([g[1] for g in geo])
    out["rstar_u"] = np.array([g[2] for g in geo])
    zoom = []
    for i in range(S):
        cx = np.clip(out["zmin_x"][i] if np.isfinite(out["zmin_x"][i]) else 0.0, -XMAX + 1.5, XMAX - 1.5)
        cu = np.clip(out["zmin_u"][i] if np.isfinite(out["zmin_u"][i]) else 0.0, -UMAX + 4.0, UMAX - 4.0)
        zx, zu = np.meshgrid(np.linspace(cx - 1.5, cx + 1.5, 121), np.linspace(cu - 4.0, cu + 4.0, 121))
        zoom.append((zx.ravel(), zu.ravel()))
    ZX, ZU = np.stack([z[0] for z in zoom]), np.stack([z[1] for z in zoom])
    _, _, juz = ev(P, jnp.asarray(ZX), jnp.asarray(ZU), s0)
    prof = dict(eq_r=r, eq_u=us, eq_Ju=jue, eq_Jx=jxe, eq_Ju_true=plant.Ju(r, us), eq_Jx_true=plant.Jx(r, us),
                map_x=xg.reshape(ngrid, ngrid), map_u=ug.reshape(ngrid, ngrid),
                map_Ju=jug.reshape(S, ngrid, ngrid).astype(np.float32), map_Ju_true=plant.Ju(xg, ug).reshape(ngrid, ngrid),
                zoom_x=ZX.reshape(S, 121, 121), zoom_u=ZU.reshape(S, 121, 121),
                zoom_Ju=np.asarray(juz).reshape(S, 121, 121).astype(np.float32))
    return out, prof


# ------------------------------------------------------------------------------------------------ #
# 6. The experiment: configurations, resumable I/O                                                 #
# ------------------------------------------------------------------------------------------------ #
LOSSES = {"J_x only": (1.0, 0.0), "J_u only": (0.0, 1.0), "J_x + J_u": (1.0, 1.0)}


def level1_configs(T=3000, T_extra=1000):
    """14 configurations. T = main training length, T_extra = the extra phase of the 4000-step schedules.
    two-stage = MSE for T steps, then MSE + Jacobian loss for T_extra steps (fresh Adam moments at the switch)."""
    C = [dict(name="MSE (3000)", group="MSE", loss="MSE", schedule="3000", norm="layer", labels=None,
              phases=[("mse", 0, 0, T)]),
         dict(name="MSE (3000+1000)", group="MSE", loss="MSE", schedule="3000+1000", norm="layer", labels=None,
              phases=[("mse", 0, 0, T), ("mse", 0, 0, T_extra)]),
         dict(name="static norm, MSE", group="kink-free", loss="MSE", schedule="3000", norm="static", labels=None,
              phases=[("mse", 0, 0, T)]),
         dict(name="no norm, MSE", group="kink-free", loss="MSE", schedule="3000", norm="none", labels=None,
              phases=[("mse", 0, 0, T)])]
    for lname, (wx, wu) in LOSSES.items():
        C += [dict(name=f"{lname} (3000)", group=lname, loss=lname, schedule="3000", norm="layer", labels="est",
                   phases=[("sob", wx, wu, T)]),
              dict(name=f"{lname} (4000)", group=lname, loss=lname, schedule="4000", norm="layer", labels="est",
                   phases=[("sob", wx, wu, T + T_extra)]),
              dict(name=f"{lname} (two-stage)", group=lname, loss=lname, schedule="two-stage", norm="layer", labels="est",
                   phases=[("mse", 0, 0, T), ("sob", wx, wu, T_extra)])]
    C.append(dict(name="J_x + J_u, TRUE labels (3000)", group="true labels", loss="J_x + J_u", schedule="3000",
                  norm="layer", labels="true", phases=[("sob", 1.0, 1.0, T)]))
    return C


def out_root(name="level1_out"):
    if os.environ.get("L1_OUT_ROOT"):
        return os.environ["L1_OUT_ROOT"]
    base = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
    return os.path.join(base, name)


def append_csv(path, rows):
    if not rows:
        return
    new = not os.path.exists(path)
    cols = list(rows[0].keys())
    if not new:
        with open(path) as f:
            cols = next(csv.reader(f))
    with open(path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerows(rows)


def safe(name):
    return "".join(ch if ch.isalnum() else "_" for ch in name).strip("_")


In [ ]:
%%writefile dpc_core.py
"""dpc_core.py - Differentiable Predictive Control (DPC) THROUGH the Level 1 surrogates.

Pipeline (the user's DPC example, applied to the LayerNorm-kink study):
  1. Identify the plant: retrain the Level 1 surrogates F(x, u) exactly as in level1_core (same data, init, labels,
     draws), 10 seeds per model.
  2. Train a GRU controller by unrolling it through the FULL surrogate x_{k+1} = F(x_k, u_k) (no linear extraction,
     so the controller sees the kink), with the LQR-type loss and the horizon curriculum of the example.
  3. Test every controller on the TRUE plant x+ = 1.03 x + 0.01 u. Reference = the same controller trained through the
     true plant ("oracle"); the headline number is the true-plant cost gap.

Changes from the example, each needed for this study: rollout through the whole network (not W, V after LayerNorm);
input bounded to |u| <= 10 by u = 10 tanh(head) (the surrogates only saw |u| <= 10 and the plant's actuator is limited
there); plain JAX instead of flax.nnx so the 10 seeds train together (same GRU equations, optax.adamw); per seed every
model's controller starts from the same weights and trains on the same 1000 initial states (paired comparisons).
"""
import os
import time
import numpy as np
import jax
import jax.numpy as jnp
import optax
import level1_core as L

tmap = jax.tree_util.tree_map

# ------------------------------------------------------------------------------------------------ #
# 0. Settings (the example's values)                                                               #
# ------------------------------------------------------------------------------------------------ #
CONFIG = dict(N=200, Q_x=5.0, R_u=0.1, Q_f=50.0)          # LQR-type loss and test horizon
CURRICULUM = [(5, 1000), (10, 2000), (20, 2000), (50, 1000), (100, 1000), (200, 2000)]   # (horizon, epochs)
HIDDEN = 64                                               # GRU width
CTRL_LR = 1e-3                                            # optax.adamw (default weight decay 1e-4, as in the example)
N_X0 = 1000                                               # training initial states per controller
X0_RANGE = 3.0                                            # x0 ~ U[-3, 3] (controllable range with |u| <= 10: |x| < 3.33)
U_BOUND = 10.0                                            # u = 10 tanh(head)
X0_EVAL = np.array(sorted({s * v for v in (0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0) for s in (-1, 1)}))
N_TEST = 1000                                             # "typical" test starts, x0 ~ U[-3, 3], same for every seed
DIVERGE = 10.0                                            # |x| above this anywhere = diverged
SETTLED = 0.05                                            # |x_N| above this at the end = not regulated
LQR_K = 10.29                                             # analytic LQR gain for (1.03, 0.01, Q=5, R=0.1)

# The models compared. 'surrogate' names a Level 1 configuration (level1_configs or wave_configs).
MODELS = [dict(name="true plant (oracle)", short="oracle", kind="true", group="oracle"),
          dict(name="LN, MSE (3000)", short="LN MSE", surrogate="MSE (3000)", group="MSE"),
          dict(name="LN, MSE (12000)", short="LN MSE\n12000", surrogate="MSE (12000)", group="MSE long"),
          dict(name="LN, J_x + J_u (3000)", short="LN slope\nfrom step 0", surrogate="J_x + J_u (3000)", group="J_x + J_u"),
          dict(name="LN, two-stage", short="LN\ntwo-stage", surrogate="J_x + J_u (two-stage)", group="two-stage"),
          dict(name="no norm, MSE", short="no\nnorm", surrogate="no norm, MSE", group="kink-free"),
          dict(name="static norm, MSE", short="static\nnorm", surrogate="static norm, MSE", group="kink-free")]

# Optimiser-side control (2026-10-23): the same LN MSE surrogate, but DPC aggregates per-start gradients with a bounded vote
# (robust_grad 'clip'). If this rescues the trapped seeds, the failure is the exploding per-trajectory gradient, and this is
# the generic "fix the optimiser" baseline. Not in MODELS by default: add it in the notebook (ROBUST_MODELS) for the A100 run.
ROBUST_MODELS = [dict(name="LN, MSE (3000), clipped grads", short="LN MSE\nclipped", surrogate="MSE (3000)", group="MSE robust",
                      aggregate="clip", clip=3.0),
                 dict(name="LN, MSE (3000), unit grads", short="LN MSE\nunit vote", surrogate="MSE (3000)", group="MSE robust",
                      aggregate="unit")]

# Optimiser baselines (Task 3, 2026-11-03): does a standard optimiser trick already fix the trap? Same LN MSE surrogate,
# same controller inits and starts. Not in MODELS by default: the notebook trains them when RUN_OPT = True.
#   aggregate 'global' : clip the AVERAGED gradient to norm <= clip (optax.clip_by_global_norm)
#   aggregate 'mom'    : median of means: starts split into `groups` contiguous groups, mean per group, coordinate-wise median
#   batch b            : b starts drawn without replacement each epoch (key 40000 + s + 1000 rep, fold_in epoch)
#   lr                 : AdamW learning rate (default CTRL_LR)
#   max_horizon        : curriculum stops at this horizon (stages with N > max_horizon dropped); the test stays N = 200
OPT_MODELS = [dict(name="LN, MSE (3000), global clip 3", short="LN MSE\nglobal clip", surrogate="MSE (3000)", group="MSE opt",
                   aggregate="global", clip=3.0),
              dict(name="LN, MSE (3000), mom", short="LN MSE\nmedian of means", surrogate="MSE (3000)", group="MSE opt",
                   aggregate="mom", groups=10),
              dict(name="LN, MSE (3000), batch 100", short="LN MSE\nbatch 100", surrogate="MSE (3000)", group="MSE opt",
                   batch=100),
              dict(name="LN, MSE (3000), lr 1e-4", short="LN MSE\nlr 1e-4", surrogate="MSE (3000)", group="MSE opt", lr=1e-4),
              dict(name="LN, MSE (3000), N <= 50", short="LN MSE\nN <= 50", surrogate="MSE (3000)", group="MSE opt",
                   max_horizon=50),
              dict(name="LN, MSE (3000), N <= 100", short="LN MSE\nN <= 100", surrogate="MSE (3000)", group="MSE opt",
                   max_horizon=100)]
BATCH_KEY = 40_000                                        # minibatch draws: PRNGKey(40000 + s + 1000 rep), fold_in(epoch)


def surrogate_configs(T=3000, T_extra=1000, T_long=12000):
    """name -> Level 1 configuration, with the step counts scaled for QUICK runs."""
    C = {c["name"]: c for c in L.level1_configs(T, T_extra)}
    long = dict(name="MSE (12000)", group="MSE", loss="MSE", schedule=str(T_long), norm="layer", labels=None,
                phases=[("mse", 0, 0, T_long)])
    C["MSE (12000)"] = long
    return C


def safe(name):
    return "".join(ch if ch.isalnum() else "_" for ch in name).strip("_")


def out_root(name="dpc_out"):
    """DPC_OUT_ROOT (Colab: MyDrive/dpc_out) or a local folder. arch_dir() appends the controller architecture."""
    if os.environ.get("DPC_OUT_ROOT"):
        return os.environ["DPC_OUT_ROOT"]
    base = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
    return os.path.join(base, name)


# ------------------------------------------------------------------------------------------------ #
# 1. Surrogates: retrain all seeds of one Level 1 configuration together, save the weights          #
# ------------------------------------------------------------------------------------------------ #
def _flat(tree, prefix=""):
    out = {}
    for k, v in tree.items():
        if isinstance(v, dict):
            out.update(_flat(v, prefix + k + "/"))
        else:
            out[prefix + k] = np.asarray(v)
    return out


def _unflat(d):
    out = {}
    for k, v in d.items():
        parts = k.split("/")
        cur = out
        for p in parts[:-1]:
            cur = cur.setdefault(p, {})
        cur[parts[-1]] = jnp.asarray(v)
    return out


def train_surrogates(cfg, seeds, plant, sigma, kernel, out_dir, label_dir=None, M=2048, chunk=250, lam=1.0):
    """Stacked weights P [S, ...] and static-norm scales S0 [S] for one Level 1 configuration, trained exactly as
    level1_core.run_config does (same data, init, labels, slope-check draws, Adam restart at each phase switch).
    Cached in out_dir/surrogates/<name>.npz; also returns the Level 1 metrics of the trained models."""
    os.makedirs(os.path.join(out_dir, "surrogates"), exist_ok=True)
    fn = os.path.join(out_dir, "surrogates", safe(cfg["name"]) + ".npz")
    if os.path.exists(fn):
        z = np.load(fn, allow_pickle=True)
        if list(z["seeds"]) == list(seeds):
            P = _unflat({k[2:]: z[k] for k in z.files if k.startswith("p:")})
            return P, z["S0"], {k[2:]: z[k] for k in z.files if k.startswith("m:")}
    label_dir = label_dir or out_dir
    t0 = time.time()
    datas, P0, S0, Jx_lab, Ju_lab = [], [], [], [], []
    for s in seeds:
        d = L.make_data(plant, sigma, 1000 + s)
        p = L.init_params(s)
        datas.append(d), P0.append(p)
        S0.append(L.frozen_sigma(p, d["x"], d["u"]) if cfg["norm"] == "static" else 1.0)
        if cfg.get("labels") == "est":
            lab = L.get_labels(d, s, sigma, kernel, label_dir)
            Jx_lab.append(lab["Jx"]), Ju_lab.append(lab["Ju"])
        elif cfg.get("labels") == "true":
            Jx_lab.append(plant.Jx(d["x"], d["u"])), Ju_lab.append(plant.Ju(d["x"], d["u"]))
        else:
            Jx_lab.append(np.zeros(L.N_DATA)), Ju_lab.append(np.zeros(L.N_DATA))
    D = tmap(jnp.asarray, dict(x=np.stack([d["x"] for d in datas]), u=np.stack([d["u"] for d in datas]),
                               y=np.stack([d["y"] for d in datas]), s0=np.array(S0),
                               sx=np.array([d["x"].std() for d in datas]), su=np.array([d["u"].std() for d in datas]),
                               cdf=np.tile(np.arange(1, L.N_DATA + 1) / L.N_DATA, (len(seeds), 1)),
                               Jx_lab=np.stack(Jx_lab), Ju_lab=np.stack(Ju_lab)))
    P = L.stack_trees(P0)
    KEYS = jnp.stack([jax.random.PRNGKey(10_000 + s) for s in seeds])
    lam = float(cfg.get("lam", lam))
    for kind, wx, wu, nsteps in cfg["phases"]:
        run = L.get_trainer(cfg["norm"], kind, M, chunk)
        Mo, Vo = tmap(jnp.zeros_like, P), tmap(jnp.zeros_like, P)
        for ci in range(max(1, nsteps // chunk)):
            P, Mo, Vo, KEYS = run(P, Mo, Vo, float(ci * chunk), KEYS, D, float(wx), float(wu), lam)
    jax.block_until_ready(P)
    met, _ = L.evaluate_batch(P, S0, cfg["norm"], plant, L.make_test(plant), 201, full=True)
    keep = ("rel_mse", "Ju_sup_rel", "Ju_rms_rel", "Jx_sup", "flip_area_interior", "lqr_unstable",
            "zmin_x", "zmin_u", "rstar_x", "rstar_u", "crest")
    met = {k: np.asarray(met[k], float) for k in keep if k in met}
    np.savez_compressed(fn, seeds=np.array(seeds), S0=np.array(S0), train_s=np.array(time.time() - t0),
                        **{"p:" + k: v for k, v in _flat(P).items()}, **{"m:" + k: v for k, v in met.items()})
    return P, np.array(S0), met


# ------------------------------------------------------------------------------------------------ #
# 2. Controllers. ARCH selects one; every controller has init(seed, d_x) -> c, step(c, h, x) -> (h, u), and a   #
#    hidden size (0 = memoryless: h is a dummy [B, 0] array, so rollout / phase code needs no special case).     #
#    'gru'    the example's StandaloneGRUController in plain JAX (hidden 64): the original study.               #
#    'mlp'    u = 10 tanh(MLP(x)), tanh hidden layers, memoryless: a plain policy u = pi(x), so phase plots show #
#             a function and K_eff = -d pi/dx is comparable to the LQR gain 10.29.                               #
#    'linear' u = 10 tanh((k x + b) / 10) ~ sat(-K x + c), 2 parameters per input: the whole DPC problem is a    #
#             2-D landscape that can be plotted exhaustively.                                                   #
# ------------------------------------------------------------------------------------------------ #
ARCH = "gru"                                              # default = the original study; set before training
MLP_WIDTH, MLP_DEPTH = 64, 2                              # 'mlp': hidden layers (tanh)
LIN_K0 = -5.0                                             # 'linear': initial gain u ~ LIN_K0 x (stabilising, half the LQR gain)
LIN_SCALE = 10.0                                          # 'linear' parameters are stored / LIN_SCALE so Adam's unit steps fit


def _linear_init(key, n_in, n_out, dtype):
    """flax.nnx.Linear default: lecun_normal kernel, zero bias."""
    return dict(W=jax.nn.initializers.lecun_normal()(key, (n_in, n_out), dtype), b=jnp.zeros(n_out, dtype))


def hidden_size(arch=None):
    return HIDDEN if (arch or ARCH) == "gru" else 0


def init_controller(seed, d_x=1, d_u=1, hidden=HIDDEN, dtype=jnp.float64, arch=None):
    """Controller weights from key 20000 + seed (run_keys passes seed + 1000 * repeat for repeats > 0).
    The same key gives the same GRU as before; the MLP and linear controllers draw from it too."""
    arch = arch or ARCH
    key = jax.random.PRNGKey(20_000 + int(seed))
    if arch == "gru":
        k = jax.random.split(key, 4)
        return dict(Wz=_linear_init(k[0], d_x + hidden, hidden, dtype), Wr=_linear_init(k[1], d_x + hidden, hidden, dtype),
                    Wh=_linear_init(k[2], d_x + hidden, hidden, dtype), head=_linear_init(k[3], hidden, d_u, dtype))
    if arch == "mlp":
        k = jax.random.split(key, MLP_DEPTH + 1)
        sizes = [d_x] + [MLP_WIDTH] * MLP_DEPTH + [d_u]
        return {f"L{i}": _linear_init(k[i], sizes[i], sizes[i + 1], dtype) for i in range(MLP_DEPTH + 1)}
    if arch == "linear":                                  # gain on the FIRST input (x, or x - r for tracking), 0 on the rest
        k = jnp.zeros((d_x, d_u), dtype).at[0, 0].set(LIN_K0 / LIN_SCALE)
        return dict(k=k, b=jnp.zeros(d_u, dtype))
    raise ValueError(arch)


def gru_step(c, h_prev, x):
    """h_new, u for a batch: x [B, d_x], h [B, hidden]. Same gates as the example; u = 10 tanh(head(h))."""
    lin = lambda l, a: a @ l["W"] + l["b"]
    hx = jnp.concatenate([x, h_prev], -1)
    z = jax.nn.sigmoid(lin(c["Wz"], hx))
    r = jax.nn.sigmoid(lin(c["Wr"], hx))
    h_tilde = jnp.tanh(lin(c["Wh"], jnp.concatenate([x, r * h_prev], -1)))
    h_new = (1 - z) * h_prev + z * h_tilde
    return h_new, U_BOUND * jnp.tanh(lin(c["head"], h_new))


def mlp_step(c, h, x):
    names = sorted(k for k in c if k.startswith("L"))      # L0 ... Ln
    a = x
    for nm in names[:-1]:
        a = jnp.tanh(a @ c[nm]["W"] + c[nm]["b"])
    l = c[names[-1]]
    return h, U_BOUND * jnp.tanh(a @ l["W"] + l["b"])


def linear_step(c, h, x):
    """u = 10 tanh((x K + B) / 10) with K = LIN_SCALE k, B = LIN_SCALE b: linear with gain K (u ~ K x + B) until it
    saturates at +-10. Stored parameters are k = K / 10, so an Adam step of 1e-3 moves the gain by 0.01."""
    return h, U_BOUND * jnp.tanh((x @ c["k"] + c["b"]) * LIN_SCALE / U_BOUND)


def ctrl_step(c, h, x):
    """Dispatch on the parameter tree, so a saved controller of any architecture runs with the same code."""
    if "Wz" in c:
        return gru_step(c, h, x)
    if "L0" in c:
        return mlp_step(c, h, x)
    return linear_step(c, h, x)


def policy_gain(c, x, r=None):
    """K_eff = -du/dx of a memoryless controller at x (scalar or array; r for tracking controllers)."""
    def u_of(xx):
        inp = jnp.array([xx]) if r is None else jnp.array([xx - r, r])
        return ctrl_step(c, jnp.zeros((1, 0)), inp[None])[1][0, 0]
    return -np.asarray(jax.vmap(jax.grad(u_of))(jnp.atleast_1d(jnp.asarray(x, float))))


# ------------------------------------------------------------------------------------------------ #
# 3. Dynamics the controller is trained through, and the DPC loss                                  #
# ------------------------------------------------------------------------------------------------ #
def make_dynamics(kind, norm=None, plant=None):
    """kind 'true': x+ = A x + B u. kind 'surrogate': x+ = F(x, u), the whole network (norm = layer/static/none)."""
    if kind == "true":
        A, B = float(plant.A), float(plant.B)
        return lambda p, s0, x, u: A * x + B * u
    Fb = jax.vmap(lambda p, x, u, s0: L.F_point(p, x, u, s0, norm), in_axes=(None, 0, 0, None))
    return lambda p, s0, x, u: Fb(p, x[:, 0], u[:, 0], s0)[:, None]


def rollout(c, dyn, p, s0, x0, N, hidden=None):
    """Closed loop of N steps through dyn. Returns xs [N+1, B, 1], us [N, B, 1]."""
    h0 = jnp.zeros((x0.shape[0], HIDDEN if "Wz" in c else 0), x0.dtype)

    def step(carry, _):
        h, x = carry
        h, u = ctrl_step(c, h, x)
        xn = dyn(p, s0, x, u)
        return (h, xn), (x, u)

    # Gradient checkpointing: keep only the carried (h, x) per step and recompute the step's internals in the backward
    # pass. Without it a full-size step (200 steps x 1000 starts x 10 seeds) needs ~13 GB for backpropagation.
    (h, xN), (xs, us) = jax.lax.scan(jax.checkpoint(step), (h0, x0), None, length=N)
    return jnp.concatenate([xs, xN[None]], 0), us


def dpc_loss(c, dyn, p, s0, x0, N, cfg=CONFIG):
    """The example's compute_lqr_loss: (sum_k mean(Q x_k^2 + R u_k^2) + mean(Q_f x_N^2)) / N."""
    xs, us = rollout(c, dyn, p, s0, x0, N)
    stage = jnp.sum(jnp.mean(cfg["Q_x"] * xs[:-1, :, 0] ** 2 + cfg["R_u"] * us[:, :, 0] ** 2, axis=1))
    return (stage + jnp.mean(cfg["Q_f"] * xs[-1, :, 0] ** 2)) / N


_DPC_STEPS = {}


def robust_grad(c, dyn, p, s0, x0, N, aggregate="mean", clip=1.0, groups=10):
    """Loss and gradient over the batch of starts x0 [B, 1] with a choice of aggregation over starts.
    'mean'  : the ordinary full-batch gradient (gradient of the mean loss).
    'clip'  : per-start gradients, each clipped to norm <= clip, then averaged (a bounded vote per trajectory: one start
              whose rollout runs through a locally unstable region of the surrogate cannot dominate the batch).
    'unit'  : per-start gradients normalised to unit norm, then averaged (equal vote, no threshold).
    'global': the ordinary mean gradient, rescaled to norm <= clip (optax.clip_by_global_norm): bounds the step size,
              not the vote of a single start (its direction is unchanged).
    'mom'   : median of means: the B starts are split into `groups` contiguous groups (B divisible by groups), the gradient
              is averaged within each group, and the coordinate-wise median over the group means is used.
    Per-start gradients cost one backward pass per start (vmap over starts), ~B x the memory of the mean gradient."""
    if aggregate == "mean":
        return jax.value_and_grad(dpc_loss)(c, dyn, p, s0, x0, N)
    if aggregate == "global":
        loss, g = jax.value_and_grad(dpc_loss)(c, dyn, p, s0, x0, N)
        nrm = jnp.sqrt(sum(jnp.sum(l ** 2) for l in jax.tree_util.tree_leaves(g)))
        sc = jnp.where(nrm < clip, 1.0, clip / nrm)
        return loss, tmap(lambda l: l * sc, g)
    one = lambda c, x: dpc_loss(c, dyn, p, s0, x[None], N)
    losses, G = jax.vmap(jax.value_and_grad(one), in_axes=(None, 0))(c, x0)          # [B], tree of [B, ...]
    if aggregate == "mom":
        B = x0.shape[0]
        G = tmap(lambda g: jnp.median(jnp.mean(g.reshape((groups, B // groups) + g.shape[1:]), 1), 0), G)
        return jnp.mean(losses), G
    nrm = jnp.sqrt(sum(jnp.sum(g.reshape(g.shape[0], -1) ** 2, 1) for g in jax.tree_util.tree_leaves(G)))   # [B]
    scale = jnp.minimum(1.0, clip / (nrm + 1e-30)) if aggregate == "clip" else 1.0 / (nrm + 1e-30)
    G = tmap(lambda g: jnp.mean(g * scale.reshape((-1,) + (1,) * (g.ndim - 1)), 0), G)
    return jnp.mean(losses), G


def get_dpc_trainer(key, dyn, N, n_epochs, opt, aggregate="mean", clip=1.0, groups=10):
    """n_epochs full-batch AdamW steps at horizon N, vmapped over seeds. Returns (C, O, losses[S, n_epochs]).
    aggregate: how per-start gradients are combined (see robust_grad); groups: only for 'mom'."""
    k = (key, N, n_epochs, aggregate, clip) if aggregate != "mom" else (key, N, n_epochs, aggregate, clip, groups)
    if k in _DPC_STEPS:
        return _DPC_STEPS[k]

    def one(c, o, p, s0, x0):
        def st(carry, _):
            c, o = carry
            loss, g = (robust_grad(c, dyn, p, s0, x0, N, aggregate, clip) if aggregate != "mom"
                       else robust_grad(c, dyn, p, s0, x0, N, aggregate, clip, groups))
            upd, o = opt.update(g, o, c)
            return (optax.apply_updates(c, upd), o), loss
        (c, o), losses = jax.lax.scan(st, (c, o), None, length=n_epochs)
        return c, o, losses

    f = jax.jit(jax.vmap(one, in_axes=(0, 0, 0, 0, 0)))
    _DPC_STEPS[k] = f
    return f


def get_dpc_trainer_batch(key, dyn, N, n_epochs, opt, batch, aggregate="mean", clip=1.0, groups=10):
    """As get_dpc_trainer, but each epoch uses `batch` starts drawn without replacement from the run's x0: the draw of
    epoch e is jax.random.choice(fold_in(bkey, e), B, (batch,), replace=False), bkey = PRNGKey(40000 + s + 1000 rep).
    Returns f(C, O, P, S0, X0, BKEYS, epoch0) -> (C, O, losses)."""
    k = ("batch", key, N, n_epochs, batch, aggregate, clip, groups)
    if k in _DPC_STEPS:
        return _DPC_STEPS[k]

    def one(c, o, p, s0, x0, bkey, epoch0):
        def st(carry, e):
            c, o = carry
            idx = jax.random.choice(jax.random.fold_in(bkey, e), x0.shape[0], (batch,), replace=False)
            loss, g = robust_grad(c, dyn, p, s0, x0[idx], N, aggregate, clip, groups)
            upd, o = opt.update(g, o, c)
            return (optax.apply_updates(c, upd), o), loss
        (c, o), losses = jax.lax.scan(st, (c, o), epoch0 + jnp.arange(n_epochs))
        return c, o, losses

    f = jax.jit(jax.vmap(one, in_axes=(0, 0, 0, 0, 0, 0, None)))
    _DPC_STEPS[k] = f
    return f


def run_keys(seed, rep):
    """Random keys of one DPC training run. rep 0 = the original run (controller init 20000 + s, starts 30000 + s);
    rep r > 0 = a fresh controller initialisation AND a fresh draw of the 1000 training starts."""
    return 20_000 + int(seed) + 1000 * int(rep), 30_000 + int(seed) + 1000 * int(rep)


def train_controllers(model, P, S0, seeds, plant, curriculum=CURRICULUM, n_x0=N_X0, log_every=250, dtype=jnp.float64,
                      verbose=True, reps=(0,)):
    """Train one controller (architecture ARCH) per (seed, repeat) through this model's dynamics, all together (vmapped).
    P, S0 are per seed; they are tiled over the repeats. Returns (C [len(reps) * S, ...], curve rows); the run order is
    repeat-major: index = k * S + i for repeat reps[k] and seed seeds[i]."""
    norm = model.get("norm")
    dyn = make_dynamics(model["kind"] if model.get("kind") == "true" else "surrogate", norm, plant)
    cast = lambda a: jnp.asarray(a, dtype) if jnp.issubdtype(jnp.asarray(a).dtype, jnp.floating) else a
    pairs = [(s, r) for r in reps for s in seeds]
    P = tmap(lambda a: jnp.concatenate([cast(a)] * len(reps), 0), P)
    S0 = jnp.asarray(np.concatenate([np.asarray(S0)] * len(reps)), dtype)
    X0 = jnp.stack([jax.random.uniform(jax.random.PRNGKey(run_keys(s, r)[1]), (n_x0, 1), dtype, -X0_RANGE, X0_RANGE)
                    for s, r in pairs])
    C = L.stack_trees([init_controller(run_keys(s, r)[0] - 20_000, dtype=dtype) for s, r in pairs])
    opt = optax.adamw(model.get("lr", CTRL_LR))
    O = jax.vmap(opt.init)(C)
    if model.get("max_horizon"):
        curriculum = [(N, e) for N, e in curriculum if N <= model["max_horizon"]]
    if model.get("batch"):
        BKEYS = jnp.stack([jax.random.PRNGKey(BATCH_KEY + int(s) + 1000 * int(r)) for s, r in pairs])
    curves, epoch, t0 = [], 0, time.time()
    for N, n_ep in curriculum:
        done = 0
        while done < n_ep:
            n = min(log_every, n_ep - done)
            if model.get("batch"):
                step = get_dpc_trainer_batch((model["name"], str(dtype), ARCH), dyn, N, n, opt, model["batch"],
                                             model.get("aggregate", "mean"), model.get("clip", 3.0), model.get("groups", 10))
                C, O, losses = step(C, O, P, S0, X0, BKEYS, epoch)
            elif model.get("aggregate") == "mom":
                step = get_dpc_trainer((model["name"], str(dtype), ARCH), dyn, N, n, opt, "mom", model.get("clip", 3.0),
                                       model.get("groups", 10))
                C, O, losses = step(C, O, P, S0, X0)
            else:
                step = get_dpc_trainer((model["name"], str(dtype), ARCH), dyn, N, n, opt, model.get("aggregate", "mean"), model.get("clip", 3.0))
                C, O, losses = step(C, O, P, S0, X0)
            losses = np.asarray(losses)
            done += n
            epoch += n
            for i, (s, r) in enumerate(pairs):
                curves.append(dict(model=model["name"], seed=s, rep=r, epoch=epoch, horizon=N, loss=float(losses[i, -1])))
        if verbose:
            last = np.array([r["loss"] for r in curves[-len(pairs):]])
            print(f"    horizon {N:3d}: {n_ep:5d} epochs | loss on its model: median {np.median(last):.4g}, "
                  f"max {np.max(last):.4g} | {time.time() - t0:.0f}s", flush=True)
    return C, curves


# ------------------------------------------------------------------------------------------------ #
# 4. Test on the TRUE plant                                                                        #
# ------------------------------------------------------------------------------------------------ #
def true_lqr_cost(xs, us, cfg=CONFIG):
    """The example's true_lqr_cost: (sum over steps of Q x^2 + R u^2, + Q_f x_N^2), mean over starts, / (N+1)."""
    stage = cfg["Q_x"] * xs[:-1] ** 2 + cfg["R_u"] * us ** 2
    return float((stage.sum(0) + cfg["Q_f"] * xs[-1] ** 2).mean() / xs.shape[0])


def per_start_cost(xs, us, cfg=CONFIG):
    """Unnormalised cost of each trajectory: sum_k (Q x_k^2 + R u_k^2) + Q_f x_N^2. xs [N+1, B], us [N, B]."""
    return (cfg["Q_x"] * xs[:-1] ** 2 + cfg["R_u"] * us ** 2).sum(0) + cfg["Q_f"] * xs[-1] ** 2


def lqr_saturated(plant, x0, N=CONFIG["N"], K=LQR_K):
    """Reference: analytic LQR u = clip(-K x, +-10) on the true plant."""
    xs, us = [np.asarray(x0, float)], []
    for _ in range(N):
        u = np.clip(-K * xs[-1], -U_BOUND, U_BOUND)
        us.append(u)
        xs.append(plant.A * xs[-1] + plant.B * u)
    return np.stack(xs), np.stack(us)


def _starts(n_runs, dtype):
    xe = jnp.asarray(np.repeat(X0_EVAL[None, :, None], n_runs, 0), dtype)
    xt = jnp.asarray(np.repeat(np.random.default_rng(12345).uniform(-X0_RANGE, X0_RANGE, (1, N_TEST, 1)), n_runs, 0), dtype)
    return xe, xt


def _roller(dyn, N=CONFIG["N"]):
    return jax.jit(jax.vmap(lambda c, p, s0, x0: rollout(c, dyn, p, s0, x0, N), in_axes=(0, 0, 0, 0)))


def evaluate_controllers(model, C, P, S0, seeds, plant, dtype=jnp.float64, kink=None, reps=(0,)):
    """Closed loops on the TRUE plant (and, for comparison, through the controller's own model) for the 22 evaluation
    starts and the 1000 typical starts, for every (seed, repeat) run in C (repeat-major order, as train_controllers).
    Returns (rows, {rep: trajectories})."""
    norm = model.get("norm")
    true_dyn = make_dynamics("true", plant=plant)
    own_dyn = true_dyn if model.get("kind") == "true" else make_dynamics("surrogate", norm, plant)
    cast = lambda a: jnp.asarray(a, dtype) if jnp.issubdtype(jnp.asarray(a).dtype, jnp.floating) else a
    pairs = [(s, r) for r in reps for s in seeds]
    P = tmap(lambda a: jnp.concatenate([cast(a)] * len(reps), 0), P)
    S0 = jnp.asarray(np.concatenate([np.asarray(S0)] * len(reps)), dtype)
    xe, xt = _starts(len(pairs), dtype)
    rt, ro = _roller(true_dyn), _roller(own_dyn)
    XE, UE = (np.asarray(a, float)[..., 0] for a in rt(C, P, S0, xe))          # [runs, N+1, 22], [runs, N, 22]
    XT, UT = (np.asarray(a, float)[..., 0] for a in rt(C, P, S0, xt))
    XO, UO = (np.asarray(a, float)[..., 0] for a in ro(C, P, S0, xt))          # same controller, its own model
    rows = []
    for i, (s, rep) in enumerate(pairs):
        J = per_start_cost(XE[i], UE[i])
        fin = np.abs(XE[i, -1])
        div = ~np.isfinite(XE[i]).all(0) | (np.nanmax(np.abs(XE[i]), 0) > DIVERGE)
        row = dict(model=model["name"], group=model["group"], seed=s, rep=rep,
                   cost_test_true=true_lqr_cost(XT[i], UT[i]), cost_test_own=true_lqr_cost(XO[i], UO[i]),
                   n_diverged=int(div.sum()), n_not_settled=int(((fin > SETTLED) & ~div).sum()),
                   final_abs_max=float(np.nanmax(fin)), final_abs_median=float(np.nanmedian(fin)),
                   test_not_settled=float(np.mean(np.abs(XT[i, -1]) > SETTLED)))
        for j, x0 in enumerate(X0_EVAL):
            row[f"J[{x0:+g}]"] = float(J[j])
        k = seeds.index(s)
        if kink is not None and np.isfinite(kink["rstar_x"][k]):
            zx, zu, rx, ru = (kink[q][k] for q in ("zmin_x", "zmin_u", "rstar_x", "rstar_u"))
            inside = ((XE[i, :-1] - zx) / rx) ** 2 + ((UE[i] - zu) / ru) ** 2 <= 1.0
            row["kink_share"] = float(inside.mean())
            row["kink_share_small"] = float(inside[:, np.abs(X0_EVAL) <= 0.2].mean())
        rows.append(row)
    trajs = {}
    for r in reps:
        idx = [i for i, (s, rr) in enumerate(pairs) if rr == r]
        trajs[r] = dict(XE=XE[idx].astype(np.float32), UE=UE[idx].astype(np.float32), seeds=np.array(seeds), x0=X0_EVAL)
    return rows, trajs


# ------------------------------------------------------------------------------------------------ #
# 5. The experiment, resumable per (model, repeat)                                                 #
# ------------------------------------------------------------------------------------------------ #
def append_rows(path, rows):
    """Append rows; if they bring columns the file lacks (e.g. kink_share after the oracle's rows), widen the file."""
    if rows and os.path.exists(path):
        import pandas as pd
        old = pd.read_csv(path)
        if set(rows[0]) - set(old.columns):
            pd.concat([old, pd.DataFrame(rows)], ignore_index=True).to_csv(path, index=False)
            return
    L.append_csv(path, rows)


def _read_runs(path):
    import pandas as pd
    df = pd.read_csv(path)
    if "rep" not in df:
        df["rep"] = 0                                   # files from before repeats existed = repeat 0
    df["rep"] = df["rep"].fillna(0).astype(int)
    return df


def done_runs(path, seeds):
    """{(model, rep)} with every seed finished."""
    if not os.path.exists(path):
        return set()
    df = _read_runs(path)
    return {(m, int(r)) for (m, r), g in df.groupby(["model", "rep"]) if set(seeds) <= set(g.seed)}


def rep_file(folder, name, rep):
    """rep 0 keeps the original file name, so a run made before repeats existed is reused as repeat 0."""
    return os.path.join(folder, safe(name) + ("" if rep == 0 else f"__rep{rep}") + ".npz")


def load_controllers(out_dir, name, rep):
    fn = rep_file(os.path.join(out_dir, "controllers"), name, rep)
    if not os.path.exists(fn):
        return None, None
    z = np.load(fn)
    return _unflat({k[2:]: z[k] for k in z.files if k.startswith("c:")}), list(z["seeds"])


def run_dpc(models, seeds, plant, sigma, kernel, out_dir, surr_cfgs, label_dir=None, curriculum=CURRICULUM,
            n_x0=N_X0, dtype=jnp.float64, M=2048, chunk=250, lam=1.0, reps=(0,), sur_dir=None):
    """sur_dir: folder whose surrogates/ (and labels) are reused; default out_dir. With ARCH != 'gru' pass the GRU study's
    folder as sur_dir and its arch_dir() as out_dir, so every architecture controls the SAME surrogates."""
    sur_dir = sur_dir or out_dir
    os.makedirs(os.path.join(out_dir, "controllers"), exist_ok=True)
    os.makedirs(os.path.join(out_dir, "trajectories"), exist_ok=True)
    res, cur = os.path.join(out_dir, "dpc_results.csv"), os.path.join(out_dir, "dpc_curves.csv")
    done = done_runs(res, seeds)
    todo = [(m["name"], r) for m in models for r in reps if (m["name"], r) not in done]
    print(f"{len(models)} models x {len(seeds)} seeds x repeats {list(reps)} -> {out_dir}  "
          f"({len(models) * len(reps) - len(todo)} model-repeats already done, skipped)")
    surr_rows = []
    for m in models:
        t0 = time.time()
        if m.get("kind") == "true":
            P, S0, met = L.stack_trees([L.init_params(s) for s in seeds]), np.ones(len(seeds)), None   # placeholder, unused
            m = dict(m, norm=None)
        else:
            cfg = surr_cfgs[m["surrogate"]]
            P, S0, met = train_surrogates(cfg, seeds, plant, sigma, kernel, sur_dir, label_dir, M, chunk, lam)
            m = dict(m, norm=cfg["norm"])
            for i, s in enumerate(seeds):
                surr_rows.append(dict(model=m["name"], seed=s, **{k: float(v[i]) for k, v in met.items()}))
        missing = [r for r in reps if (m["name"], r) not in done]
        if not missing:
            continue
        print(f"  DPC through '{m['name']}', repeats {missing}", flush=True)
        C, curves = train_controllers(m, P, S0, seeds, plant, curriculum, n_x0, dtype=dtype, reps=missing)
        rows, trajs = evaluate_controllers(m, C, P, S0, seeds, plant, dtype, met if m.get("norm") == "layer" else None, missing)
        for r in rows:
            r["train_s_per_seed"] = (time.time() - t0) / (len(seeds) * len(missing))
        append_rows(res, rows)
        append_rows(cur, curves)
        for k, rep in enumerate(missing):
            np.savez_compressed(rep_file(os.path.join(out_dir, "trajectories"), m["name"], rep), **trajs[rep])
            Ck = tmap(lambda a: a[k * len(seeds):(k + 1) * len(seeds)], C)
            np.savez_compressed(rep_file(os.path.join(out_dir, "controllers"), m["name"], rep), seeds=np.array(seeds),
                                **{"c:" + kk: v for kk, v in _flat(Ck).items()})
        med = lambda k: np.median([r[k] for r in rows])
        print(f"  -> {m['name']:24s} | true-plant test cost {med('cost_test_true'):.4g} (own model {med('cost_test_own'):.4g}) "
              f"| diverged {sum(r['n_diverged'] for r in rows)} | not settled {sum(r['n_not_settled'] for r in rows)} "
              f"| {time.time() - t0:.0f}s", flush=True)
    if surr_rows:
        import pandas as pd
        pd.DataFrame(surr_rows).to_csv(os.path.join(out_dir, "surrogates.csv"), index=False)


# ------------------------------------------------------------------------------------------------ #
# 6. Diagnostic: the reference controller run through each surrogate                              #
# ------------------------------------------------------------------------------------------------ #
def run_diagnostic(models, seeds, plant, out_dir, reps=(0,), dtype=jnp.float64, sur_dir=None):
    """For every surrogate model and repeat: take the controller trained through the TRUE plant (same seed and repeat)
    and run it through the surrogate. Two questions:
      1. Does the surrogate misbehave even under a good controller?  (oracle_on_model_* vs oracle_on_true)
      2. Which controller does the surrogate rate better: the good one, or the one DPC found through it?
         own_on_model < oracle_on_model -> the model prefers its own (possibly bad) controller: the MODEL misleads DPC.
         own_on_model > oracle_on_model -> a better controller existed for this model and DPC did not find it: an
         OPTIMISATION trap.
    Rows go to dpc_diag.csv (resumable per model and repeat)."""
    import pandas as pd
    path = os.path.join(out_dir, "dpc_diag.csv")
    done = set()
    if os.path.exists(path):
        d = pd.read_csv(path)
        done = {(m, int(r)) for (m, r), g in d.groupby(["model", "rep"]) if set(seeds) <= set(g.seed)}
    R = _read_runs(os.path.join(out_dir, "dpc_results.csv"))
    rows = []
    for m in models:
        if m.get("kind") == "true":
            continue
        cfg = surrogate_configs()[m["surrogate"]]
        P, S0, met, sseeds = load_surrogate(sur_dir or out_dir, m["surrogate"])
        idx = [sseeds.index(s) for s in seeds]
        P = tmap(lambda a: jnp.asarray(np.asarray(a)[idx], dtype), P)
        S0 = jnp.asarray(np.asarray(S0)[idx], dtype)
        roll = _roller(make_dynamics("surrogate", cfg["norm"], plant))
        for rep in reps:
            if (m["name"], rep) in done:
                continue
            Co, cs = load_controllers(out_dir, "true plant (oracle)", rep)
            if Co is None:
                continue
            Co = tmap(lambda a: jnp.asarray(np.asarray(a)[[cs.index(s) for s in seeds]], dtype), Co)
            xe, xt = _starts(len(seeds), dtype)
            XT, UT = (np.asarray(a, float)[..., 0] for a in roll(Co, P, S0, xt))
            XE, UE = (np.asarray(a, float)[..., 0] for a in roll(Co, P, S0, xe))
            for i, s in enumerate(seeds):
                g = R[(R.model == m["name"]) & (R.rep == rep) & (R.seed == s)]
                o = R[(R.model == "true plant (oracle)") & (R.rep == rep) & (R.seed == s)]
                fin = np.abs(XE[i, -1])
                div = ~np.isfinite(XE[i]).all(0) | (np.nanmax(np.abs(XE[i]), 0) > DIVERGE)
                rows.append(dict(model=m["name"], seed=s, rep=rep,
                                 oracle_on_model=true_lqr_cost(XT[i], UT[i]),
                                 oracle_on_true=float(o.cost_test_true.iloc[0]) if len(o) else np.nan,
                                 own_on_model=float(g.cost_test_own.iloc[0]) if len(g) else np.nan,
                                 own_on_true=float(g.cost_test_true.iloc[0]) if len(g) else np.nan,
                                 oracle_on_model_diverged=int(div.sum()),
                                 oracle_on_model_not_settled=int(((fin > SETTLED) & ~div).sum()),
                                 oracle_on_model_offset=float(np.nanmedian(np.abs(XE[i, -20:]).mean(0)))))
    if rows:
        append_rows(path, rows)
    return pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()


def load_surrogate(out_dir, name):
    """(P, S0, metrics) of a saved surrogate configuration."""
    z = np.load(os.path.join(out_dir, "surrogates", safe(name) + ".npz"), allow_pickle=True)
    return (_unflat({k[2:]: z[k] for k in z.files if k.startswith("p:")}), z["S0"],
            {k[2:]: z[k] for k in z.files if k.startswith("m:")}, list(z["seeds"]))


def arch_dir(out_dir, arch=None):
    """The GRU study keeps its original folder; other architectures get a sibling folder (<out>_mlp, <out>_linear)
    so results never mix. Surrogates and labels are shared: pass sur_dir=out_dir to the training functions."""
    arch = arch or ARCH
    return out_dir if arch == "gru" else out_dir.rstrip("/") + "_" + arch


def linear_landscape(dyn, p, s0, x0, N, ks, bs, cfg=CONFIG):
    """DPC loss of the LINEAR controller u ~ k x + b (gain k in plant units, saturating at +-10) on a (k, b) grid, through `dyn` (a surrogate or the true
    plant), for one seed: the whole optimisation problem DPC faces, as a 2-D map. Returns loss [len(ks), len(bs)]."""
    ks, bs = jnp.asarray(ks, x0.dtype), jnp.asarray(bs, x0.dtype)

    def one(k, b):
        c = dict(k=jnp.full((1, 1), k / LIN_SCALE, x0.dtype), b=jnp.full((1,), b / LIN_SCALE, x0.dtype))
        return dpc_loss(c, dyn, p, s0, x0, N, cfg)
    f = jax.jit(jax.vmap(jax.vmap(one, in_axes=(None, 0)), in_axes=(0, None)))
    return np.asarray(f(ks, bs))


def gradient_audit(dyn_sur, dyn_true, p, s0, x0, N, ks, b=0.0, cfg=CONFIG):
    """Test of Corollary U5 on the linear controller u ~ k x + b: dL/dk through the SURROGATE and through the TRUE plant
    at the same gains k, same starts, same horizon. Where the two have opposite signs, DPC through the surrogate is pushed
    away from the direction that improves the true cost. Returns dict(k, g_sur, g_true, L_sur, L_true)."""
    ks = jnp.asarray(ks, x0.dtype)

    def loss(k, dyn):
        c = dict(k=jnp.full((1, 1), k / LIN_SCALE, x0.dtype), b=jnp.full((1,), b / LIN_SCALE, x0.dtype))
        return dpc_loss(c, dyn, p, s0, x0, N, cfg)
    vg = lambda dyn: jax.jit(jax.vmap(jax.value_and_grad(lambda k: loss(k, dyn))))
    Ls, gs = vg(dyn_sur)(ks)
    Lt, gt = vg(dyn_true)(ks)
    return dict(k=np.asarray(ks), g_sur=np.asarray(gs) / LIN_SCALE, g_true=np.asarray(gt) / LIN_SCALE,
                L_sur=np.asarray(Ls), L_true=np.asarray(Lt))


In [ ]:
%%writefile dpc_report.py
"""dpc_report.py - tables, paired statistics and figures for the DPC-through-the-surrogate experiment."""
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Ellipse
from scipy.stats import wilcoxon, spearmanr
import dpc_core as D

# One colour per model, fixed (validated order: adjacent pairs separable in colour-vision deficiency). LayerNorm + MSE
# is the neutral grey baseline (the problem), the oracle is black. Identity is also in every tick label / legend.
COL = {"true plant (oracle)": "#0b0b0b", "LN, MSE (3000)": "#6f6e69", "LN, MSE (12000)": "#eda100",
       "LN, J_x + J_u (3000)": "#1baf7a", "LN, two-stage": "#6250d6", "no norm, MSE": "#eb6834",
       "static norm, MSE": "#2a78d6"}
LS = {"true plant (oracle)": "-", "LN, MSE (3000)": "-", "LN, MSE (12000)": "--", "LN, J_x + J_u (3000)": "-.",
      "LN, two-stage": "-", "no norm, MSE": ":", "static norm, MSE": "--"}
ORDER = [m["name"] for m in D.MODELS]
SHORT = {m["name"]: m["short"] for m in D.MODELS}
KINKFREE = ("static norm, MSE", "no norm, MSE")
SMALL, LARGE = 0.2, 1.0          # |x0| <= 0.2 = small starts (begin inside or next to the kink); |x0| >= 1 = large

plt.rcParams.update({"font.size": 9, "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#8a8984",
                     "axes.titlesize": 9, "axes.titleweight": "bold", "legend.frameon": False, "savefig.dpi": 160})


def _save(fig, out_dir, name):
    d = os.path.join(out_dir, "figs")
    os.makedirs(d, exist_ok=True)
    fig.savefig(os.path.join(d, name + ".png"), bbox_inches="tight")
    fig.savefig(os.path.join(d, name + ".pdf"), bbox_inches="tight")


def _rep(df):
    if "rep" not in df:
        df["rep"] = 0
    df["rep"] = df["rep"].fillna(0).astype(int)
    return df


FAIL = 10.0     # a run "fails" when its true-plant cost on the typical starts is more than 10 % above the oracle's


def load(out_dir):
    R = _rep(pd.read_csv(os.path.join(out_dir, "dpc_results.csv"))).drop_duplicates(["model", "seed", "rep"], keep="last")
    C = _rep(pd.read_csv(os.path.join(out_dir, "dpc_curves.csv"))).drop_duplicates(["model", "seed", "rep", "epoch"], keep="last")
    sp = os.path.join(out_dir, "surrogates.csv")
    S = pd.read_csv(sp) if os.path.exists(sp) else pd.DataFrame()
    return R, C, S


def traj(out_dir, model, rep=0):
    fn = D.rep_file(os.path.join(out_dir, "trajectories"), model, rep)
    return np.load(fn) if os.path.exists(fn) else None


def reps_of(out_dir):
    """Repeats that have trajectories on disk (0 = the original run)."""
    import glob, re
    found = {0}
    for f in glob.glob(os.path.join(out_dir, "trajectories", "*__rep*.npz")):
        found.add(int(re.search(r"__rep(\d+)\.npz$", f).group(1)))
    return sorted(found)


def per_start(out_dir, models=None, tail=20):
    """One row per (model, seed, start x0) from the saved true-plant trajectories:
      J        total cost  sum_k (Q x_k^2 + R u_k^2) + Q_f x_N^2   (what the true plant pays)
      offset   x_inf = mean of x over the last `tail` steps (a steady error left by a biased model)
      J_dyn    the same cost about the controller's own resting point (x - x_inf, u - u_inf): the transient only
      ratio, ratio_dyn = J / J_oracle on the same seed and start (1 = as good as the oracle)."""
    rows = []
    for rep in reps_of(out_dir):
        for m in models or ORDER:
            T = traj(out_dir, m, rep)
            if T is None:
                continue
            XE, UE = T["XE"].astype(float), T["UE"].astype(float)
            for i, s in enumerate(T["seeds"]):
                x, u = XE[i], UE[i]
                xi, ui = x[-tail:].mean(0), u[-tail:].mean(0)
                J = D.per_start_cost(x, u)
                Jd = D.per_start_cost(x - xi, u - ui)
                for j, x0 in enumerate(T["x0"]):
                    rows.append(dict(model=m, seed=int(s), rep=rep, x0=float(x0), J=J[j], J_dyn=Jd[j], offset=xi[j],
                                     final=x[-1, j]))
    P = pd.DataFrame(rows)
    o = P[P.model == "true plant (oracle)"].set_index(["seed", "rep", "x0"])
    key = list(zip(P.seed, P.rep, P.x0))
    P["ratio"] = P.J.values / o.loc[key, "J"].values
    P["ratio_dyn"] = P.J_dyn.values / o.loc[key, "J_dyn"].values
    return P


def add_kink_share(R, out_dir):
    """Share of the 22 evaluation trajectories' steps (x_k, u_k) inside each LayerNorm surrogate's closed-form kink ellipse,
    computed from the saved trajectories (runs before the column fix lost it from dpc_results.csv)."""
    sp = os.path.join(out_dir, "surrogates.csv")
    if not os.path.exists(sp):
        return R
    S = pd.read_csv(sp)
    rows = []
    for rep in reps_of(out_dir):
        for m in [m for m in ORDER if m in set(S.model)]:
            T = traj(out_dir, m, rep)
            if T is None:
                continue
            for i, s in enumerate(T["seeds"]):
                g = S[(S.model == m) & (S.seed == s)]
                if not len(g) or not np.isfinite(g.rstar_x.iloc[0]):
                    continue
                g = g.iloc[0]
                ins = ((T["XE"][i, :-1] - g.zmin_x) / g.rstar_x) ** 2 + ((T["UE"][i] - g.zmin_u) / g.rstar_u) ** 2 <= 1
                rows.append(dict(model=m, seed=int(s), rep=rep, kink_share=float(ins.mean()),
                                 kink_share_small=float(ins[:, np.abs(T["x0"]) <= SMALL].mean())))
    if not rows:
        return R
    R = R.drop(columns=[c for c in ("kink_share", "kink_share_small") if c in R])
    return R.merge(pd.DataFrame(rows), on=["model", "seed", "rep"], how="left")


def add_gaps(R, P):
    """Per (model, seed): cost gaps vs the oracle controller of the same seed, in %.
      gap_test   true-plant cost on the 1000 typical starts (the example's headline number)
      gap_small  median over small starts (|x0| <= 0.2) of J / J_oracle - 1     (offset included)
      gap_small_dyn  the same for the transient only (offset removed)
      gap_large  median over large starts (|x0| >= 1)
      gap_worst  worst start, transient only (offset removed; the offset has its own column)
      offset     median |x_inf| over the 22 starts (steady error)"""
    R = R.copy()
    o = R[R.model == "true plant (oracle)"].set_index(["seed", "rep"]).cost_test_true
    R["gap_test"] = 100 * (R.cost_test_true / o.loc[list(zip(R.seed, R.rep))].values - 1)
    a = np.abs(P.x0)
    g = lambda mask, col, f: (P[mask].groupby(["model", "seed", "rep"])[col].agg(f) - 1) * 100
    add = pd.DataFrame({"gap_small": g(a <= SMALL, "ratio", "median"), "gap_small_dyn": g(a <= SMALL, "ratio_dyn", "median"),
                        "gap_large": g(a >= LARGE, "ratio", "median"), "gap_worst": g(a > 0, "ratio_dyn", "max"),
                        "offset": P.assign(o=P.offset.abs()).groupby(["model", "seed", "rep"]).o.median()})
    return R.merge(add.reset_index(), on=["model", "seed", "rep"], how="left")


def lqr_reference(plant, out_dir=None):
    """Analytic LQR with saturation on the same 1000 typical starts: the true-plant cost a perfect linear design gets."""
    x0 = np.random.default_rng(12345).uniform(-D.X0_RANGE, D.X0_RANGE, D.N_TEST)
    xs, us = D.lqr_saturated(plant, x0)
    return D.true_lqr_cost(xs, us)


def _q(s):
    s = pd.Series(s).dropna()
    return f"{np.median(s):.3g} [{np.percentile(s, 25):.2g}, {np.percentile(s, 75):.2g}]" if len(s) else "-"


def summary_table(R, out_dir):
    rows = []
    for m in [m for m in ORDER if m in set(R.model)]:
        g = R[R.model == m]
        row = {"model": m, "runs (seeds x repeats)": f"{g.seed.nunique()} x {g.rep.nunique()}",
               "true-plant cost (typical starts)": _q(g.cost_test_true),
               "cost on its own model": _q(g.cost_test_own)}
        if m != "true plant (oracle)":
            for k, lab in (("gap_test", "gap typical %"), ("gap_small", "gap small starts %"),
                           ("gap_small_dyn", "gap small, transient only %"), ("gap_large", "gap large starts %"),
                           ("gap_worst", "gap worst start, transient only %")):
                row[lab] = _q(g[k])
                if k == "gap_worst":
                    row["gap worst start: worst run %"] = f"{g[k].max():.3g}"
            f = g.gap_test > FAIL
            per_seed = f.groupby(g.seed).agg(["sum", "count"])
            row[f"runs failed (gap > {FAIL:g} %)"] = f"{int(f.sum())}/{len(g)}"
            row["seeds failing in some / every repeat"] = (f"{int((per_seed['sum'] > 0).sum())} / "
                                                           f"{int((per_seed['sum'] == per_seed['count']).sum())}  "
                                                           f"(seeds {', '.join(str(s) for s in per_seed.index[per_seed['sum'] > 0])})")
        row["steady offset |x_inf| (median)"] = _q(g.offset)
        row["diverged / not settled (of 22 starts x seeds)"] = f"{int(g.n_diverged.sum())} / {int(g.n_not_settled.sum())}"
        if "kink_share" in g and g.kink_share.notna().any():
            row["share of steps inside the kink (all / small starts)"] = f"{g.kink_share.median():.2f} / {g.kink_share_small.median():.2f}"
        rows.append(row)
    T = pd.DataFrame(rows)
    os.makedirs(os.path.join(out_dir, "tables"), exist_ok=True)
    T.to_csv(os.path.join(out_dir, "tables", "dpc_summary.csv"), index=False)
    return T


COMPARISONS = [
    ("Does the kink hurt?", "LN, MSE (3000)", "static norm, MSE"),
    ("Does the kink hurt?", "LN, MSE (3000)", "no norm, MSE"),
    ("Does the slope loss help?", "LN, two-stage", "LN, MSE (3000)"),
    ("Does the slope loss help?", "LN, J_x + J_u (3000)", "LN, MSE (3000)"),
    ("Does it reach kink-free?", "LN, two-stage", "static norm, MSE"),
    ("Does it reach kink-free?", "LN, two-stage", "no norm, MSE"),
    ("Just train longer?", "LN, MSE (12000)", "LN, MSE (3000)"),
    ("Just train longer?", "LN, two-stage", "LN, MSE (12000)"),
]


def paired_table(R, out_dir, metrics=("gap_test", "gap_small_dyn", "gap_large", "gap_worst", "offset")):
    """Per seed (median over its repeats), A and B share the surrogate's data/init and the controllers' inits/initial states.
    Median of A - B
    (gaps are in %, so the difference is in percentage points), seeds where A is lower (better), Wilcoxon p."""
    rows = []
    Rs = R.groupby(["model", "seed"])[list(metrics)].median().reset_index()       # one value per seed: median over repeats
    for q, a, b in COMPARISONS:
        if a not in set(Rs.model) or b not in set(Rs.model):
            continue
        A, B = Rs[Rs.model == a].set_index("seed"), Rs[Rs.model == b].set_index("seed")
        s = sorted(set(A.index) & set(B.index))
        row = {"question": q, "A": a, "B": b, "n": len(s)}
        for m in metrics:
            x, y = A.loc[s, m].values, B.loc[s, m].values
            try:
                p = wilcoxon(x, y).pvalue if len(s) >= 5 and np.any(x != y) else np.nan
            except ValueError:
                p = np.nan
            row[f"{m}: median A-B"] = f"{np.median(x - y):.3g}"
            row[f"{m}: A better"] = f"{int(np.sum(x < y))}/{len(s)}"
            row[f"{m}: p"] = f"{p:.3g}" if np.isfinite(p) else "-"
        rows.append(row)
    T = pd.DataFrame(rows)
    os.makedirs(os.path.join(out_dir, "tables"), exist_ok=True)
    T.to_csv(os.path.join(out_dir, "tables", "dpc_paired.csv"), index=False)
    return T


def _strip(ax, R, col, order, scale="symlog", band=True, f=1.0):
    rng = np.random.default_rng(0)
    ref = R[R.model.isin(KINKFREE)][col] * f
    if band and len(ref):
        ax.axhspan(ref.min(), ref.max(), color="#b3b2ab", alpha=0.25, lw=0, label="range of kink-free models")
    for i, m in enumerate(order):
        v = R[R.model == m][col].values * f
        ax.scatter(i + rng.uniform(-0.15, 0.15, len(v)), v, s=24, color=COL[m], edgecolor="white", linewidth=0.6, zorder=3)
        ax.plot([i - 0.3, i + 0.3], [np.median(v)] * 2, color="#0b0b0b", lw=1.6, zorder=4)
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels([SHORT[m] for m in order], fontsize=7)
    if scale == "symlog":
        ax.set_yscale("symlog", linthresh=1)
        lo = np.nanmin(R[R.model.isin(order)][col].values * f)
        ax.set_ylim(bottom=-1.5 if lo > -1 else 1.5 * lo)
    elif scale == "log":
        ax.set_yscale("log")


def fig_headline(R, out_dir, lqr_cost=None):
    """Figure D1: one dot per seed, bar = median, grey band = range of the two kink-free models."""
    order = [m for m in ORDER if m in set(R.model) and m != "true plant (oracle)"]
    panels = [("gap_test", "true-plant cost gap vs oracle, typical starts (%)", "symlog"),
              ("gap_large", "cost gap, large starts |x0| >= 1 (median, %)", "symlog"),
              ("gap_small_dyn", "cost gap, small starts |x0| <= 0.2, transient only (median, %)", "symlog"),
              ("gap_worst", "cost gap, worst of the 22 starts, transient only (%)", "symlog"),
              ("offset", "steady offset |x_inf| (median over starts)", "log"),
              ("cost_ratio_own", "cost on its own model / on the true plant (typical starts)", "log")]
    R = R.assign(cost_ratio_own=R.cost_test_own / R.cost_test_true)
    fig, axes = plt.subplots(2, 3, figsize=(16, 7.6))
    for ax, (col, title, sc) in zip(axes.ravel(), panels):
        _strip(ax, R, col, order, sc)
        ax.set_title(title, fontsize=8.5)
        if col.startswith("gap"):
            ax.axhline(0, color="#8a8984", lw=0.8)
        if col == "gap_test" and lqr_cost is not None:
            o = R[R.model == "true plant (oracle)"].cost_test_true.median()
            yl = 100 * (lqr_cost / o - 1)
            ax.axhline(yl, color="#0b0b0b", ls="--", lw=1, label="analytic LQR with saturation")
            ax.set_ylim(bottom=min(ax.get_ylim()[0], 1.5 * yl if yl < 0 else ax.get_ylim()[0]))
        if col == "cost_ratio_own":
            ax.axhline(1, color="#8a8984", lw=0.8)
    axes[0, 0].legend(fontsize=7, loc="upper right")
    fig.suptitle("DPC trained through each model, tested on the true plant (one dot per run = seed x repeat, bar = median; "
                 "gap = cost / cost of the same DPC trained through the true plant - 1). Axes: linear within +-1 %, log beyond.",
                 fontsize=9, y=1.0)
    fig.tight_layout()
    _save(fig, out_dir, "figD1_headline")
    return fig


def worst_seed(R, model="LN, MSE (3000)", col="gap_small_dyn"):
    g = R[R.model == model].groupby("seed")[col].median()
    return int(g.idxmax()) if len(g) else 0


def fig_closed_loop(out_dir, seed, starts=(-2.0, -0.5, -0.05, 0.05, 0.5, 2.0), models=None):
    """Figure D2 (the example's plot_closed_loop, one column per model): x(t), |x(t)| (log) and u(t) on the TRUE plant.
    Thin black = the oracle controller from the same start."""
    models = [m for m in (models or ORDER) if m != "true plant (oracle)" and traj(out_dir, m) is not None]
    O = traj(out_dir, "true plant (oracle)")
    fig, axes = plt.subplots(3, len(models), figsize=(3.2 * len(models), 8.0), sharex=True, sharey="row", squeeze=False)
    cmap = plt.get_cmap("coolwarm")
    for j, m in enumerate(models):
        T = traj(out_dir, m)
        i = list(T["seeds"]).index(seed)
        io = list(O["seeds"]).index(seed)
        for k, x0 in enumerate(starts):
            jj = int(np.argmin(np.abs(T["x0"] - x0)))
            c = cmap(0.1 + 0.8 * k / max(1, len(starts) - 1))
            t = np.arange(T["XE"].shape[1])
            for r_, (y, yo) in enumerate(((T["XE"][i, :, jj], O["XE"][io, :, jj]), (np.abs(T["XE"][i, :, jj]) + 1e-9,
                                         np.abs(O["XE"][io, :, jj]) + 1e-9))):
                axes[r_, j].plot(t, yo, color="#0b0b0b", lw=0.7, alpha=0.6)
                axes[r_, j].plot(t, y, color=c, lw=1.5, label=f"x0 = {x0:g}" if j == 0 and r_ == 0 else None)
            axes[2, j].step(t[:-1], T["UE"][i, :, jj], where="post", color=c, lw=1.3)
        axes[0, j].set_title(m, fontsize=8.5, color=COL[m])
        axes[1, j].set_yscale("log")
        axes[2, j].set_xlabel("step k")
    axes[0, 0].set_ylabel("state x on the true plant")
    axes[1, 0].set_ylabel("|x| (log): offsets and slow decay")
    axes[2, 0].set_ylabel("control u")
    axes[0, 0].legend(fontsize=6.5, loc="upper right")
    fig.suptitle(f"Seed {seed}: DPC controllers trained through each model, run on the TRUE plant "
                 "(thin black = DPC trained through the true plant, same start)", fontsize=9.5, y=1.0)
    fig.tight_layout()
    _save(fig, out_dir, f"figD2_closed_loop_seed{seed}")
    return fig


def fig_norm(out_dir, models=None):
    """Figure D3 (the example's plot_norm_comparison): median over seeds of the mean |x(t)| on the true plant, separately
    for small starts (inside / next to the kink) and large starts."""
    models = [m for m in (models or ORDER) if traj(out_dir, m) is not None]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), sharey=False)
    for ax, (mask_f, title) in zip(axes, ((lambda a: a <= SMALL, f"small starts, |x0| <= {SMALL}"),
                                          (lambda a: a >= LARGE, f"large starts, |x0| >= {LARGE}"))):
        for m in models:
            Ts = [t for t in (traj(out_dir, m, r) for r in reps_of(out_dir)) if t is not None]
            sel = mask_f(np.abs(Ts[0]["x0"]))
            n = np.concatenate([np.abs(t["XE"][:, :, sel]).mean(2) for t in Ts], 0)     # [runs, N+1]
            t = np.arange(n.shape[1])
            ax.semilogy(t, np.median(n, 0) + 1e-9, color=COL[m], ls=LS[m], lw=2 if m != "true plant (oracle)" else 1.2,
                        label=m)
            ax.fill_between(t, np.percentile(n, 25, 0) + 1e-9, np.percentile(n, 75, 0) + 1e-9, color=COL[m], alpha=0.08, lw=0)
        ax.set_title(title)
        ax.set_xlabel("step k")
        ax.set_ylabel("mean |x| over starts (median over runs, band = IQR)")
    h, l = axes[0].get_legend_handles_labels()
    fig.suptitle("Closed-loop regulation on the TRUE plant", fontsize=9.5, y=1.02)
    fig.tight_layout()
    fig.legend(h, l, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=4, fontsize=7.5)
    _save(fig, out_dir, "figD3_state_norm")
    return fig


def fig_phase(out_dir, seed, plant, models=("LN, MSE (3000)", "LN, two-stage", "static norm, MSE"), wx=0.6, wu=6.0, n=161,
              sur_dir=None):
    """Figure D4: where the closed loop runs. True-plant trajectories (x_k, u_k) from the small and medium starts, drawn
    over each surrogate's input-gain ratio (model dF/du / true B; blue = wrong sign) with its kink ellipse."""
    import jax.numpy as jnp
    import level1_core as L
    cfgs = {m["name"]: m for m in D.MODELS}
    models = [m for m in models if traj(out_dir, m) is not None]
    fig, axes = plt.subplots(1, len(models), figsize=(4.6 * len(models) + 1, 4.4), squeeze=False, constrained_layout=True)
    norm_ = TwoSlopeNorm(vmin=-3, vcenter=1, vmax=5)
    im = None
    for j, m in enumerate(models):
        ax = axes[0, j]
        P, S0, met, seeds = D.load_surrogate(sur_dir or out_dir, cfgs[m]["surrogate"])
        i = seeds.index(seed)
        sur = D.surrogate_configs()[cfgs[m]["surrogate"]]
        p = L.unstack_tree(P, i)
        X, U = np.meshgrid(np.linspace(-wx, wx, n), np.linspace(-wu, wu, n))
        _, _, ju = L.get_evaluator(sur["norm"])(L.stack_trees([p]), jnp.asarray(X.ravel()[None]), jnp.asarray(U.ravel()[None]),
                                                jnp.asarray([S0[i]]))
        G = np.asarray(ju)[0].reshape(X.shape) / plant.B
        im = ax.pcolormesh(X, U, np.clip(G, -3, 5), cmap="RdBu_r", norm=norm_, shading="auto")
        if np.any(G < 0):
            ax.contour(X, U, G, levels=[0], colors="#0b0b0b", linewidths=0.7)
        if np.isfinite(met["rstar_x"][i]):
            ax.add_patch(Ellipse((met["zmin_x"][i], met["zmin_u"][i]), 2 * met["rstar_x"][i], 2 * met["rstar_u"][i],
                                 fill=False, color="#0b0b0b", lw=1.0, ls="--"))
        T = traj(out_dir, m)
        k = list(T["seeds"]).index(seed)
        for jj, x0 in enumerate(T["x0"]):
            if abs(x0) <= 0.5:
                ax.plot(T["XE"][k, :-1, jj], T["UE"][k, :, jj], "-", color="#0b0b0b", lw=0.8, alpha=0.8)
                ax.plot(T["XE"][k, 0, jj], T["UE"][k, 0, jj], "o", ms=3, color="#0b0b0b")
        ax.set_xlim(-wx, wx); ax.set_ylim(-wu, wu)
        ax.set_title(f"{m}, seed {seed}", fontsize=8.5)
        ax.set_xlabel("x"); ax.grid(False)
        if j == 0:
            ax.set_ylabel("u")
    cb = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.01)
    cb.set_label("model / true input gain (1 = correct, < 0 = wrong sign)", fontsize=8)
    fig.suptitle("Where the closed loop runs: true-plant trajectories (x_k, u_k) from starts |x0| <= 0.5 (dots = start), over the "
                 "surrogate's input gain; dashed ellipse = closed-form kink", fontsize=9)
    _save(fig, out_dir, f"figD4_phase_seed{seed}")
    return fig


def fig_curves(C, out_dir):
    """Figure D5: DPC loss on the controller's own model during training (median over seeds, band = IQR)."""
    fig, ax = plt.subplots(figsize=(11, 4.4))
    for m in [m for m in ORDER if m in set(C.model)]:
        st = C[C.model == m].groupby("epoch").loss
        x = st.median().index.values
        ax.semilogy(x, st.median().values, color=COL[m], ls=LS[m], lw=1.8, label=m)
        ax.fill_between(x, st.quantile(0.25).values, st.quantile(0.75).values, color=COL[m], alpha=0.1, lw=0)
    e = 0
    for N, n in D.CURRICULUM[:-1]:
        e += n
        ax.axvline(e, color="#8a8984", lw=0.7, ls=":")
    ax.set_xlabel("epoch (dotted = horizon increases: 5, 10, 20, 50, 100, 200)")
    ax.set_ylabel("DPC loss on its own model")
    ax.legend(fontsize=7, ncol=2)
    ax.set_title("DPC training curves (the loss is divided by the horizon, so it drops at each horizon increase)")
    fig.tight_layout()
    _save(fig, out_dir, "figD5_training_curves")
    return fig


def fig_kink_vs_gap(R, S, out_dir, col="gap_small_dyn"):
    """Figure D6: does the size of the surrogate's kink predict the closed-loop damage? x = peak input-gain error of the
    surrogate (Level 1 metric), y = cost gap; one dot per (model, seed)."""
    if not len(S):
        return None
    M = R.merge(S[["model", "seed", "Ju_sup_rel"]], on=["model", "seed"])
    fig, ax = plt.subplots(figsize=(7.5, 4.6))
    for m in [m for m in ORDER if m in set(M.model)]:
        g = M[M.model == m]
        ax.scatter(g.Ju_sup_rel, g[col], s=30, color=COL[m], edgecolor="white", linewidth=0.6, label=m, zorder=3)
    ax.set_xscale("log"); ax.set_yscale("symlog", linthresh=1)
    lo = np.nanmin(M[col].values)
    ax.set_ylim(bottom=-1.5 if lo > -1 else 1.5 * lo)
    ax.axhline(0, color="#8a8984", lw=0.8)
    rho = spearmanr(M.Ju_sup_rel, M[col])[0] if len(M) > 3 else np.nan
    ax.set_xlabel("surrogate's peak input-gain error  sup |dF/du - B| / B")
    ax.set_ylabel(f"closed-loop {col} (%)")
    ax.set_title(f"Kink size vs closed-loop damage (Spearman rho = {rho:.2f} over all {len(M)} model-seed pairs)")
    ax.legend(fontsize=7, loc="upper left")
    fig.tight_layout()
    _save(fig, out_dir, "figD6_kink_vs_gap")
    return fig


def fig_robustness(R, out_dir, col="gap_test"):
    """Figure D7: is a failure a property of the surrogate (fails in every repeat) or of one unlucky DPC training (fails in
    one repeat)? Rows = seeds, columns = models; colour = median cost gap over the repeats (log); text = failed repeats."""
    from matplotlib.colors import LogNorm
    models = [m for m in ORDER if m in set(R.model) and m != "true plant (oracle)"]
    seeds = sorted(R.seed.unique())
    G = np.full((len(seeds), len(models)), np.nan)
    T = [["" for _ in models] for _ in seeds]
    for j, m in enumerate(models):
        for i, s in enumerate(seeds):
            g = R[(R.model == m) & (R.seed == s)][col]
            if len(g):
                G[i, j] = np.median(g)
                T[i][j] = f"{int((g > FAIL).sum())}/{len(g)}"
    fig, ax = plt.subplots(figsize=(1.6 * len(models) + 2.5, 0.42 * len(seeds) + 1.8))
    im = ax.imshow(np.clip(np.abs(G), 1e-2, 1e5), cmap="Greys", norm=LogNorm(1e-2, 1e5), aspect="auto")
    for i in range(len(seeds)):
        for j in range(len(models)):
            if T[i][j]:
                dark = np.abs(G[i, j]) > 30
                ax.text(j, i, f"{G[i, j]:.2g} %\n{T[i][j]} failed", ha="center", va="center", fontsize=6.5,
                        color="white" if dark else "#0b0b0b", fontweight="bold" if T[i][j][0] != "0" else "normal")
    ax.set_xticks(range(len(models))); ax.set_xticklabels([SHORT[m] for m in models], fontsize=7.5)
    ax.set_yticks(range(len(seeds))); ax.set_yticklabels([f"seed {s}" for s in seeds], fontsize=7.5)
    ax.grid(False)
    cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cb.set_label("median true-plant cost gap over repeats (%, log)", fontsize=8)
    ax.set_title(f"Robustness to the controller's training: cost gap per seed (median over repeats) and how many repeats "
                 f"failed (gap > {FAIL:g} %)", fontsize=9)
    fig.tight_layout()
    _save(fig, out_dir, "figD7_robustness")
    return fig


def fig_diagnostic(Dg, R, out_dir):
    """Figure D8: the reference controller run through each surrogate.
    Left: its cost on the surrogate / on the true plant (1 = the surrogate treats a good controller like the plant does).
    Right: cost on the surrogate of the controller DPC found through it / of the reference controller. Below 1: the
    surrogate rates its own (possibly bad) controller better, so the MODEL misled DPC. Above 1: a better controller existed
    for this surrogate and DPC missed it (an OPTIMISATION trap). Crosses = runs that failed on the true plant."""
    if not len(Dg):
        return None
    Dg = Dg.merge(R[["model", "seed", "rep", "gap_test"]], on=["model", "seed", "rep"], how="left")
    order = [m for m in ORDER if m in set(Dg.model)]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
    rng = np.random.default_rng(0)
    for ax, (y, title) in zip(axes, ((Dg.oracle_on_model / Dg.oracle_on_true,
                                      "reference controller: cost on the surrogate / on the true plant"),
                                     (Dg.own_on_model / Dg.oracle_on_model,
                                      "on the surrogate: cost of DPC's own controller / of the reference controller"))):
        Dg = Dg.assign(y=y.values)
        for i, m in enumerate(order):
            g = Dg[Dg.model == m]
            f = g.gap_test > FAIL
            xj = i + rng.uniform(-0.18, 0.18, len(g))
            ax.scatter(xj[~f.values], g.y[~f], s=22, color=COL[m], edgecolor="white", linewidth=0.5, zorder=3)
            ax.scatter(xj[f.values], g.y[f], s=46, color=COL[m], marker="X", edgecolor="#0b0b0b", linewidth=0.6, zorder=4,
                       label="failed on the true plant" if i == 0 else None)
        ax.axhline(1, color="#8a8984", lw=0.9)
        ax.set_yscale("log")
        ax.set_xticks(range(len(order))); ax.set_xticklabels([SHORT[m] for m in order], fontsize=7)
        ax.set_title(title, fontsize=8.5)
    axes[1].text(0.02, 0.97, "above 1: a better controller existed for the model and DPC missed it (optimisation trap)\n"
                             "below 1: the model prefers DPC's controller (the model misleads)",
                 transform=axes[1].transAxes, fontsize=7, color="#52514e", va="top")
    h, l = axes[0].get_legend_handles_labels()
    if h:
        axes[0].legend(fontsize=7, loc="upper right")
    fig.suptitle("Diagnostic: the controller trained through the TRUE plant, run through each surrogate (one dot per run)",
                 fontsize=9.5, y=1.02)
    fig.tight_layout()
    _save(fig, out_dir, "figD8_diagnostic")
    return fig


# ------------------------------------------------------------------------------------------------ #
# Memoryless controllers: policy maps, effective gains, and the linear controller's loss landscape  #
# ------------------------------------------------------------------------------------------------ #
def _policy(c, xs, r=None):
    import jax.numpy as jnp
    inp = xs[:, None] if r is None else np.column_stack([xs - r, np.full_like(xs, r)])
    return np.asarray(D.ctrl_step(c, jnp.zeros((len(xs), 0)), jnp.asarray(inp))[1])[:, 0]


def policy_gain_table(out_dir, seeds, models, rep=0, xs=(0.0, 0.5, 2.0)):
    """K_eff = -d pi/dx at a few states, per model and seed (memoryless controllers). True saturated LQR: 10.29 for
    |x| < 0.97, 0 beyond (saturated)."""
    rows = []
    for m in models:
        C, cs = D.load_controllers(out_dir, m["name"], rep)
        if C is None or "Wz" in C:
            continue
        for s in seeds:
            c1 = D.tmap(lambda a: a[cs.index(s)], C)
            g = D.policy_gain(c1, list(xs))
            rows.append(dict(model=m["name"], seed=s, **{f"K_eff(x={x:g})": float(v) for x, v in zip(xs, g)},
                             u_at_0=float(_policy(c1, np.array([0.0]))[0])))
    T = pd.DataFrame(rows)
    os.makedirs(os.path.join(out_dir, "tables"), exist_ok=True)
    T.to_csv(os.path.join(out_dir, "tables", "policy_gains.csv"), index=False)
    return T.round(3)


def fig_policy_maps(out_dir, sur_dir, seeds, plant, models, rep=0, xlim=2.0, name="figD9_policy_maps"):
    """Rows = models, columns = seeds: the learned policy u = pi(x) (black) over the surrogate's input-gain ratio along it
    (coloured dots: white = right, red = too large, blue = wrong sign), the saturated LQR (grey dashed) and the kink
    half-width in x (grey band). A healthy policy is a straight line of slope -10.3 clipped at +-10."""
    import jax.numpy as jnp
    import level1_core as L
    models = [m for m in models if m.get("kind") != "true" and D.load_controllers(out_dir, m["name"], rep)[0] is not None]
    xs = np.linspace(-xlim, xlim, 801)
    fig, axes = plt.subplots(len(models), len(seeds), figsize=(max(2.6 * len(seeds), 7.5), 2.4 * len(models)), squeeze=False,
                             sharex=True, sharey=True)
    norm_ = TwoSlopeNorm(vmin=-3, vcenter=1, vmax=5)
    sc = None
    for a, m in enumerate(models):
        C, cs = D.load_controllers(out_dir, m["name"], rep)
        if "Wz" in C:
            continue
        P, S0, met, ss = D.load_surrogate(sur_dir, m["surrogate"])
        sur = D.surrogate_configs()[m["surrogate"]]
        ev = L.get_evaluator(sur["norm"])
        for j, s in enumerate(seeds):
            ax = axes[a, j]
            c1 = D.tmap(lambda a_: a_[cs.index(s)], C)
            u = _policy(c1, xs)
            i = ss.index(s)
            _, _, ju = ev(L.stack_trees([L.unstack_tree(P, i)]), jnp.asarray(xs[None]), jnp.asarray(u[None]), jnp.asarray([S0[i]]))
            g = np.asarray(ju)[0] / plant.B
            ax.plot(xs, np.clip(-D.LQR_K * xs, -10, 10), color="#8a8984", ls="--", lw=1)
            ax.plot(xs, u, color="#0b0b0b", lw=1.4, zorder=2)
            bad = np.abs(g - 1) > 0.5                                   # colour only where the surrogate is off by > 50 %
            sc = ax.scatter(xs[bad], u[bad], c=np.clip(g[bad], -3, 5), cmap="RdBu_r", norm=norm_, s=14, zorder=3,
                            edgecolor="none")
            if met is not None and np.isfinite(met["rstar_x"][i]):
                ax.axvspan(met["zmin_x"][i] - met["rstar_x"][i], met["zmin_x"][i] + met["rstar_x"][i], color="#f0efec", zorder=0)
            k0 = float(D.policy_gain(c1, [0.0])[0])
            ax.set_title(f"{m['short'].replace(chr(10), ' ')}, seed {s}\nK_eff(0) = {k0:.1f}", fontsize=7.5)
            ax.set_ylim(-11, 11)
            ax.tick_params(labelsize=6.5)
            if a == len(models) - 1:
                ax.set_xlabel("x", fontsize=7.5)
            if j == 0:
                ax.set_ylabel("u = pi(x)", fontsize=7.5)
    fig.colorbar(sc, ax=axes, fraction=0.015, pad=0.01).set_label("surrogate / true input gain along the policy", fontsize=8)
    fig.suptitle("Learned regulation policies (black) vs the saturated LQR (grey dashed, gain 10.29). Coloured dots = points of the "
                 "policy where the surrogate's input gain is off by > 50 % (red too large, blue wrong sign); grey band = kink half-width",
                 fontsize=9, y=1.0 + 0.3 / fig.get_figheight())
    _save(fig, out_dir, name)
    return fig


def fig_linear_landscape(out_dir, sur_dir, plant, seeds, models=("LN, MSE (3000)", "LN, two-stage"), N=200, n_x0=200,
                         ks=np.linspace(-30, 2, 97), cs=np.linspace(-3, 3, 25), name="figD10_linear_landscape"):
    """DPC loss of the 2-parameter controller u ~ K x + c on a (K, c) grid, through the surrogate (left of each pair) and
    through the true plant (right), horizon N, starts x0 evenly spaced in [-3, 3]. Star = minimum; dot = where DPC's linear
    controller ended (if one was trained in out_dir). The true plant's optimum is K ~ -14 to -17 (saturated LQR)."""
    import jax.numpy as jnp
    import level1_core as L
    from matplotlib.colors import LogNorm
    cfgs = {m["name"]: m for m in D.MODELS}
    models = [m for m in models if m in cfgs]
    x0 = jnp.linspace(-3, 3, n_x0)[:, None]
    dynT = D.make_dynamics("true", plant=plant)
    fig, axes = plt.subplots(len(seeds), 2 * len(models), figsize=(3.4 * 2 * len(models), 3.4 * len(seeds)), squeeze=False)
    im = None
    for a, s in enumerate(seeds):
        for b, m in enumerate(models):
            spec = cfgs[m]
            P, S0, met, ss = D.load_surrogate(sur_dir, spec["surrogate"])
            i = ss.index(s)
            p1 = L.unstack_tree(P, i)
            dynS = D.make_dynamics("surrogate", D.surrogate_configs()[spec["surrogate"]]["norm"], plant)
            LS = D.linear_landscape(dynS, p1, jnp.asarray(S0[i]), x0, N, ks, cs)
            LT = D.linear_landscape(dynT, p1, jnp.asarray(S0[i]), x0, N, ks, cs)
            lo = np.nanmin([LS.min(), LT.min()])
            hi = lo * 30
            C, cseeds = D.load_controllers(out_dir, m, 0)
            for q, (Lm, ttl) in enumerate(((LS, "through the surrogate"), (LT, "through the true plant"))):
                ax = axes[a, 2 * b + q]
                im = ax.pcolormesh(ks, cs, np.clip(Lm, lo, hi).T, cmap="viridis_r", norm=LogNorm(lo, hi), shading="auto")
                ax.contour(ks, cs, Lm.T, levels=[lo * 1.01, lo * 1.05, lo * 1.2], colors="white", linewidths=0.6)
                ki, ci = np.unravel_index(np.nanargmin(Lm), Lm.shape)
                ax.plot(ks[ki], cs[ci], "*", ms=12, mfc="white", mec="#0b0b0b")
                if C is not None and "k" in C and s in cseeds:
                    c1 = D.tmap(lambda a_: a_[cseeds.index(s)], C)
                    ax.plot(float(c1["k"][0, 0]) * D.LIN_SCALE, float(c1["b"][0]) * D.LIN_SCALE, "o", ms=7, color="#eb6834", mec="white")
                ax.set_title(f"{m}, seed {s}: {ttl}\nmin at K = {ks[ki]:.1f}, c = {cs[ci]:.1f}, loss {Lm[ki, ci]:.3g}", fontsize=7.5)
                ax.tick_params(labelsize=6.5)
                ax.set_xlabel("gain K  (u ~ K x + c)", fontsize=7.5)
                if 2 * b + q == 0:
                    ax.set_ylabel("offset c", fontsize=7.5)
                ax.grid(False)
    fig.colorbar(im, ax=axes, fraction=0.015, pad=0.01).set_label("DPC loss (log; clipped at 30 x the minimum)", fontsize=8)
    fig.suptitle(f"The whole DPC problem for a linear controller, horizon {N}: loss over (K, c) through the surrogate vs the true "
                 "plant. White contours = 1 %, 5 %, 20 % above the minimum; star = minimum; orange dot = where DPC ended.",
                 fontsize=9, y=1.0 + 0.3 / fig.get_figheight())
    fig.subplots_adjust(hspace=0.55, wspace=0.3, top=1 - 0.9 / fig.get_figheight())
    _save(fig, out_dir, name)
    return fig


def fig_gradient_audit(sur_dir, plant, seeds, models=("LN, MSE (3000)", "LN, two-stage", "static norm, MSE"), N=200, n_x0=200,
                       ks=np.linspace(-30, 2, 161), name="figD11_gradient_audit"):
    """Corollary U5 in closed loop: dL/dK of the linear controller through the surrogate vs through the true plant, as a
    function of the gain K (b = 0). Where the curves have opposite signs (shaded), gradient descent through the surrogate
    moves K the wrong way. Grey dashed = the true plant's optimum K; saturated LQR sits near -14 to -17 at horizon 200."""
    import jax.numpy as jnp
    import level1_core as L
    cfgs = {m["name"]: m for m in D.MODELS}
    models = [m for m in models if m in cfgs]
    x0 = jnp.linspace(-3, 3, n_x0)[:, None]
    dynT = D.make_dynamics("true", plant=plant)
    fig, axes = plt.subplots(len(seeds), len(models), figsize=(4.2 * len(models), 2.8 * len(seeds)), squeeze=False, sharex=True)
    rows = []
    for a, s in enumerate(seeds):
        for b, m in enumerate(models):
            spec = cfgs[m]
            P, S0, met, ss = D.load_surrogate(sur_dir, spec["surrogate"])
            i = ss.index(s)
            p1 = L.unstack_tree(P, i)
            dynS = D.make_dynamics("surrogate", D.surrogate_configs()[spec["surrogate"]]["norm"], plant)
            G = D.gradient_audit(dynS, dynT, p1, jnp.asarray(S0[i]), x0, N, ks)
            ax = axes[a, b]
            ax.plot(G["k"], G["g_true"], color="#0b0b0b", lw=1.6, label="through the true plant")
            ax.plot(G["k"], G["g_sur"], color=COL.get(m, "#d03b3b"), lw=1.6, ls="--", label="through the surrogate")
            wrong = np.sign(G["g_sur"]) != np.sign(G["g_true"])
            ax.fill_between(G["k"], ax.get_ylim()[0] if False else -1e9, 1e9, where=wrong, color="#fbebeb", zorder=0)
            kopt = G["k"][np.argmin(G["L_true"])]
            ksur = G["k"][np.argmin(G["L_sur"])]
            ax.axvline(kopt, color="#8a8984", ls="--", lw=1)
            ax.axvline(ksur, color=COL.get(m, "#d03b3b"), ls=":", lw=1)
            ax.axhline(0, color="#8a8984", lw=0.8)
            ax.set_yscale("symlog", linthresh=1e-2)
            lim = np.nanmax(np.abs(np.concatenate([G["g_true"], G["g_sur"]])))
            ax.set_ylim(-lim * 1.2, lim * 1.2)
            ax.set_title(f"{m}, seed {s}\nwrong-sign gradient on {100 * wrong.mean():.0f} % of K; argmin true {kopt:.1f}, surrogate {ksur:.1f}", fontsize=7.5)
            ax.tick_params(labelsize=6.5)
            if a == len(seeds) - 1:
                ax.set_xlabel("linear gain K (u ~ K x, saturating at +-10)", fontsize=7.5)
            if b == 0:
                ax.set_ylabel("dL/dK (symlog)", fontsize=7.5)
            rows.append(dict(model=m, seed=s, wrong_sign_share=float(wrong.mean()), k_opt_true=float(kopt), k_opt_sur=float(ksur)))
    axes[0, 0].legend(fontsize=7)
    fig.suptitle("Gradient audit (Corollary U5): dL/dK through the surrogate vs the true plant for the linear controller; "
                 "red shading = opposite signs", fontsize=9, y=1.0 + 0.25 / fig.get_figheight())
    fig.tight_layout()
    _save(fig, "/".join(sur_dir.split("/")), name) if False else None
    d = os.path.join(sur_dir, "figs"); os.makedirs(d, exist_ok=True)
    fig.savefig(os.path.join(d, name + ".png"), bbox_inches="tight"); fig.savefig(os.path.join(d, name + ".pdf"), bbox_inches="tight")
    return fig, pd.DataFrame(rows)


In [ ]:
import os, sys, time, shutil
try:                                   # Colab: save to Google Drive so a disconnect loses nothing
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DPC_OUT_ROOT"] = "/content/drive/MyDrive/dpc_out"
except ImportError:
    pass
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import jax.numpy as jnp
sys.path.insert(0, os.getcwd())
import importlib, level1_core as L, dpc_core as D, dpc_report as RP
importlib.reload(L); importlib.reload(D); importlib.reload(RP)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 50)
OUT = D.out_root()
os.makedirs(OUT, exist_ok=True)
print("results go to", OUT)

## Settings
`LABEL_DIR`: the slope labels take ~16 s per seed to compute. If you attach the Level 1 output as a Kaggle dataset, point
`LABEL_DIR` at the folder with `labels_seed*_sigma0.01_epan.npz` to reuse them; otherwise they are computed and cached in
the output folder.

In [ ]:
QUICK  = False                 # True: 2 seeds, short surrogates, 120 controller epochs - a ~10 minute check
ARCH   = "gru"                 # controller: "gru" (the original study), "mlp" (u = 10 tanh(MLP(x)), memoryless), "linear" (u ~ K x + c)
SEEDS  = list(range(10))       # same seeds as Level 1
DTYPE  = "float64"             # "float64" on an A100 (or CPU); "float32" on L4 / G4 / T4. Surrogates are float64 either way
SIGMA, KERNEL = 0.01, "epan"   # Level 1 data noise and label kernel
LABEL_DIR = None               # e.g. "/kaggle/input/level1-out/level1_out" to reuse the Level 1 labels
ONLY   = None                  # e.g. ["true plant (oracle)", "LN, MSE (3000)"] to run a subset
REPS   = [0, 1, 2]             # DPC training repeats; 0 = the original run (reused if already in OUT)

CURRICULUM, N_X0, SURR = D.CURRICULUM, D.N_X0, D.surrogate_configs(3000, 1000, 12000)
if QUICK:
    SEEDS = [0, 1]
    CURRICULUM = [(5, 20), (10, 20), (20, 20), (50, 20), (100, 20), (200, 20)]
    N_X0, SURR = 200, D.surrogate_configs(300, 100, 1200)
    REPS = [0, 1]
    OUT = OUT.rstrip("/") + "_quick"
if DTYPE == "float32":                                  # no TF32 shortcut: full float32 precision in every matrix product
    jax.config.update("jax_default_matmul_precision", "highest")
    OUT = OUT.rstrip("/") + "_f32"                      # keep float32 results apart from float64 ones
D.ARCH = ARCH
SUR_DIR = OUT                                           # surrogates and labels: shared by every architecture
OUT = D.arch_dir(OUT, ARCH)                             # "gru" keeps the original folder; others get <folder>_<arch>
os.makedirs(OUT, exist_ok=True)
PLANT = L.Plant(A=1.03, B=0.01, eps_nl=0.0)
MODELS = [m for m in D.MODELS if ONLY is None or m["name"] in ONLY]
print("controller:", ARCH, "| results go to", OUT, "| surrogates from", SUR_DIR)
pd.DataFrame([dict(model=m["name"], surrogate=m.get("surrogate", "- (true plant)"),
                   surrogate_training=" -> ".join(f"{k} x {n}" for k, wx, wu, n in SURR[m["surrogate"]]["phases"]) if m.get("surrogate") else "-")
              for m in MODELS])

## Train: surrogates, then one DPC controller per model and seed
For each model: retrain (or load) its 10 surrogates, train 10 GRU controllers through them together (one line per horizon
of the curriculum: the DPC loss on the controller's own model), then test them on the true plant. One summary line per
model: median true-plant cost on 1000 typical starts, the same controller's cost on its own model, and failure counts.

In [ ]:
t0 = time.time()
D.run_dpc(MODELS, SEEDS, PLANT, SIGMA, KERNEL, OUT, SURR, label_dir=LABEL_DIR, curriculum=CURRICULUM, n_x0=N_X0,
          dtype=jnp.float64 if DTYPE == "float64" else jnp.float32, reps=REPS, sur_dir=SUR_DIR)
print(f"done in {(time.time() - t0) / 60:.1f} min")

## Check 1 - the surrogates are the Level 1 models
Medians over seeds of the Level 1 metrics of the surrogates used here. With the full settings they should match the
Level 1 run: peak input-gain error 18.7 (LN MSE), 2.3 (MSE 12 000), 7.9 (J_x + J_u from step 0), 1.15 (two-stage),
0.83 (no norm), 0.47 (static norm) x B.

In [ ]:
R, CUR, SUR = RP.load(OUT)
if len(SUR):
    display(SUR.groupby("model")[["Ju_sup_rel", "Ju_rms_rel", "rel_mse", "lqr_unstable", "rstar_x", "rstar_u"]].median()
            .loc[[m for m in RP.ORDER if m in set(SUR.model)]])

## Check 2 - is the oracle controller good?
The oracle is the GRU trained through the true plant. Its true-plant cost should be close to analytic LQR with saturation
(`u = clip(-10.29 x, +-10)`) on the same 1000 starts. If it is much worse, the controller training itself (not the model)
limits every result below.

In [ ]:
LQR_COST = RP.lqr_reference(PLANT)
o = R[R.model == "true plant (oracle)"].cost_test_true
print(f"analytic LQR with saturation: {LQR_COST:.4f}   |   oracle DPC, median over seeds: {o.median():.4f} "
      f"(range {o.min():.4f} - {o.max():.4f}, i.e. {100 * (o.median() / LQR_COST - 1):+.1f} % vs LQR)")

## Summary table
Median [inter-quartile range] over seeds. **gap** = cost of the controller trained through the model / cost of the oracle
controller on the same seed and start - 1, in %, all measured on the TRUE plant (0 = as good as the oracle).
* *typical*: the 1000 starts x0 ~ U[-3, 3] (the example's headline number).
* *small starts* (|x0| <= 0.2: they begin inside or next to the kink) and *large starts* (|x0| >= 1): median over the
  evaluation starts +-{0.01, 0.02, 0.05, 0.1, 0.2} and +-{1, 1.5, 2, 2.5, 3}.
* *transient only*: the same cost measured about the controller's own resting point, so a steady offset does not count.
  The offset is its own column: |x_inf| = mean |x| over the last 20 steps, left by a model whose prediction is biased.
* *diverged*: |x| > 10 anywhere; *not settled*: |x_200| > 0.05.
* *runs failed*: runs (seed x repeat) whose typical-start gap exceeds 10 %; *seeds failing in some / every repeat*.
* *share of steps inside the kink*: fraction of the (x_k, u_k) of the 22 evaluation trajectories inside the surrogate's
  closed-form kink ellipse (LayerNorm models only).

In [ ]:
P_START = RP.per_start(OUT)
R = RP.add_gaps(R, P_START)
R = RP.add_kink_share(R, OUT)
display(RP.summary_table(R, OUT))

## Paired comparisons
For each seed, both controllers share the surrogate's data and initial weights and, within each repeat, the controller's
initial weights and training starts. Each seed's value is the median over its repeats, so the test has 10 pairs (one per
seed): median of A - B in percentage points (negative = A better), seeds where A is better, Wilcoxon p (10/10 gives
p = 0.002).

In [ ]:
display(RP.paired_table(R, OUT))

## Figure D7 - Robustness: does a failure belong to the surrogate or to one unlucky training?
Rows = seeds, columns = models. Colour and number = the median true-plant cost gap over the repeats; second line = how many
repeats failed (gap > 10 %). A seed that fails in every repeat has a surrogate DPC cannot handle; a seed that fails once
had an unlucky training.

In [ ]:
RP.fig_robustness(R, OUT); plt.show()

## Figure D8 - Diagnostic: the reference controller run through each surrogate
Left: the controller trained through the TRUE plant, run through each surrogate; its cost there / on the true plant
(1 = the surrogate judges a good controller as the plant does). Right: on each surrogate, the cost of the controller DPC
found through it / of the reference controller. Above 1 = a better controller existed for this surrogate and DPC missed it
(an optimisation trap); below 1 = the surrogate prefers DPC's controller (the model misleads). Crosses = runs that failed
on the true plant.

In [ ]:
DIAG = D.run_diagnostic(MODELS, SEEDS, PLANT, OUT, reps=REPS, dtype=jnp.float64 if DTYPE == "float64" else jnp.float32, sur_dir=SUR_DIR)
RP.fig_diagnostic(DIAG, R, OUT); plt.show()
d = DIAG.merge(R[["model", "seed", "rep", "gap_test"]], on=["model", "seed", "rep"])
d["own / reference on the model"] = d.own_on_model / d.oracle_on_model
d["reference: model / true plant"] = d.oracle_on_model / d.oracle_on_true
display(d[d.gap_test > RP.FAIL][["model", "seed", "rep", "gap_test", "reference: model / true plant",
                                  "own / reference on the model", "oracle_on_model_diverged"]].round(3))

## Memoryless controllers only: the learned policy and the DPC loss landscape
Skipped for the GRU (its output depends on the history, so there is no function u = pi(x) to draw).
**Policy map (fig D9):** u = pi(x) of every seed's controller over the surrogate's input-gain profile along that policy,
with the saturated LQR u = clip(-10.29 x) for comparison; K_eff = -d pi/dx at x = 0 per seed and model.
**Loss landscape (fig D10, "linear" only, also drawn for "mlp" as a reference):** the DPC loss of the 2-parameter
controller u ~ K x + c on a (K, c) grid, through the surrogate and through the true plant, for the seeds that fail.
If DPC on this surrogate ends at a wrong K while the landscape's minimum is right, the trap is an optimisation effect.

In [ ]:
if ARCH != "gru":
    RP.fig_policy_maps(OUT, SUR_DIR, SEEDS, PLANT, MODELS); plt.show()
    display(RP.policy_gain_table(OUT, SEEDS, MODELS))
    RP.fig_linear_landscape(OUT, SUR_DIR, PLANT, seeds=sorted(set([RP.worst_seed(R), SEEDS[0]]))); plt.show()

## Gradient audit (any ARCH; needs only the surrogates)
Corollary U5 says DPC's gradient through a kinked surrogate can point the wrong way. Test: for the 2-parameter controller
u ~ K x, compute dL/dK through the surrogate and through the true plant at the same gains and starts. Red shading = the two
disagree in sign, i.e. gradient descent through the surrogate moves K away from the true optimum. Figure D11.

In [ ]:
fig, GA = RP.fig_gradient_audit(SUR_DIR, PLANT, seeds=sorted(set([RP.worst_seed(R), SEEDS[0]]))); plt.show()
display(GA.round(3))

## Figure D1 - Headline
One dot per seed, black bar = median, grey band = range of the two kink-free models. Dashed line (top left) = analytic
LQR with saturation relative to the oracle. Bottom right: cost the controller expects on its own model / what it gets on
the true plant (1 = the model told the truth along the closed loop).

In [ ]:
RP.fig_headline(R, OUT, LQR_COST); plt.show()

## Figure D2 - Closed loops on the true plant (your `plot_closed_loop`, one column per model)
Seed where LayerNorm MSE does worst on small starts, and seed 0. Rows: x(t), |x(t)| on a log axis (shows offsets and slow
decay near zero) and u(t). Thin black = the oracle controller from the same start.

In [ ]:
for s in dict.fromkeys([RP.worst_seed(R), SEEDS[0]]):
    RP.fig_closed_loop(OUT, s); plt.show()

## Figure D3 - Convergence on the true plant (your `plot_norm_comparison`)
Median over seeds of the mean |x(t)| over the starts, small starts and large starts separately.

In [ ]:
RP.fig_norm(OUT); plt.show()

## Figure D4 - Where the closed loop runs, and what the model says there
True-plant trajectories (x_k, u_k) from the starts |x0| <= 0.5, drawn over the surrogate's input gain (model dF/du / true B:
white = correct, red = too large, blue = wrong sign, black contour = sign flip) and its closed-form kink (dashed ellipse).
If P1 holds, the LayerNorm MSE controller's trajectories linger or circle inside the kink.

In [ ]:
for s in dict.fromkeys([RP.worst_seed(R), SEEDS[0]]):
    RP.fig_phase(OUT, s, PLANT, sur_dir=SUR_DIR); plt.show()
    RP.fig_phase(OUT, s, PLANT, wx=1.5, wu=8.0, sur_dir=SUR_DIR); plt.show()      # wider window: wrong resting points away from (0, 0)

## Figure D5 - DPC training curves
The DPC loss on the controller's own model (median over seeds, band = IQR). The loss is divided by the horizon, so it drops
at each horizon increase (dotted lines). A curve that rises at long horizons means the controller could not stabilise its
own model.

In [ ]:
RP.fig_curves(CUR, OUT); plt.show()

## Figure D6 - Does the kink's size predict the closed-loop damage?
x = the surrogate's peak input-gain error (the Level 1 headline metric), y = small-start transient cost gap; one dot per
model and seed.

In [ ]:
RP.fig_kink_vs_gap(R, SUR, OUT); plt.show()

## Optimiser-side control (2026-10-23): robust per-start gradients
The trap diagnostic (local analysis, `NOTES_dpc_level1.md`, "Trap mechanism") found that on the failing LayerNorm MSE
seeds the surrogate's loss gradient at a GOOD controller is 1e13-1e35 times too large and chaotic in direction, and that a
SINGLE training start (one whose rollout runs through a region where the surrogate's slopes make the closed loop locally
unstable) carries 60-100 % of the batch gradient. The healthy surrogates' per-start gradients never exceed ~3.

**Control.** The same LayerNorm MSE surrogate, same controller inits and starts, but the per-start gradients are combined
with a bounded vote before AdamW sees them: `clip` = each start's gradient clipped to norm <= 3 (just above the healthy
maximum), `unit` = each normalised to unit norm (equal vote, no threshold). If this rescues seeds 1, 2, 8, 9 the failure is
the exploding per-trajectory gradient, and this is the "generic fix the optimiser" baseline a reviewer will ask for. It
costs one backward pass per start (`jax.vmap` over the 1000 starts), roughly 3-5x the time of a plain model on the A100.
Runs for whichever `ARCH` is set (the robust models are trained with the same controller class). Set `RUN_ROBUST = False` to skip.
`ROBUST_REPS` (default `[0]`, as before) sets the repeats; Task 3 uses `[0, 1, 2]`.

In [ ]:
RUN_ROBUST = True
ROBUST_REPS = [0]              # Task 3: [0, 1, 2]
if RUN_ROBUST:
    t0 = time.time()
    D.run_dpc(D.ROBUST_MODELS, SEEDS, PLANT, SIGMA, KERNEL, OUT, SURR, label_dir=LABEL_DIR, curriculum=CURRICULUM, n_x0=N_X0,
              dtype=jnp.float64 if DTYPE == "float64" else jnp.float32, reps=ROBUST_REPS, sur_dir=SUR_DIR)
    print(f"robust models done in {(time.time() - t0) / 60:.1f} min")
    R2, _, _ = RP.load(OUT)
    R2 = RP.add_gaps(R2, RP.per_start(OUT, models=[m["name"] for m in D.MODELS + D.ROBUST_MODELS]))
    names = ["LN, MSE (3000)"] + [m["name"] for m in D.ROBUST_MODELS] + ["LN, two-stage", "static norm, MSE"]
    T = R2[(R2.rep == 0) & R2.model.isin(names)].pivot_table(index="seed", columns="model", values="gap_test")[names]
    print("true-plant cost gap vs the oracle (%), repeat 0, per seed:")
    display(T.round(2))
    print("runs failed (gap > 10 %):", {m: int((T[m] > RP.FAIL).sum()) for m in names})

## Optimiser baselines (Task 3, 2026-11-03): does a standard optimiser trick already fix the trap?
Same LN MSE (3000) surrogates, same controller inits and training starts, one setting changed per model (`D.OPT_MODELS`):
`global clip 3` (the AVERAGED gradient clipped to norm <= 3, as `optax.clip_by_global_norm`), `mom` (median of means over
10 groups of 100 starts), `batch 100` (100 of the 1000 starts drawn per epoch, key 40000 + s + 1000 rep), `lr 1e-4`,
`N <= 50` and `N <= 100` (curriculum stopped at that horizon; the test is still 200 steps on the true plant).
Then the trap diagnostic (reference controller through the same surrogate) for every robust and optimiser model, and one
table: failed (typical-start gap > 10 %) and trapped (failed AND own controller > 1.05 x the reference on the surrogate)
out of len(SEEDS) x len(OPT_REPS) (the robust models are included when `RUN_ROBUST = True`). `RUN_OPT = False`
(default) skips the block.

In [ ]:
RUN_OPT = False
OPT_REPS = [0, 1, 2]
if RUN_OPT:
    t0 = time.time()
    D.run_dpc(D.OPT_MODELS, SEEDS, PLANT, SIGMA, KERNEL, OUT, SURR, label_dir=LABEL_DIR, curriculum=CURRICULUM, n_x0=N_X0,
              dtype=jnp.float64 if DTYPE == "float64" else jnp.float32, reps=OPT_REPS, sur_dir=SUR_DIR)
    print(f"optimiser baselines done in {(time.time() - t0) / 60:.1f} min")
    extra = (D.ROBUST_MODELS if RUN_ROBUST else []) + D.OPT_MODELS
    done = D.done_runs(os.path.join(OUT, "dpc_results.csv"), SEEDS)
    for m in extra:                                   # only finished (model, repeat) pairs, so no empty rows are cached
        reps_done = [r for r in sorted(set(OPT_REPS) | set(ROBUST_REPS)) if (m["name"], r) in done]
        D.run_diagnostic([m], SEEDS, PLANT, OUT, reps=reps_done, dtype=jnp.float64 if DTYPE == "float64" else jnp.float32,
                         sur_dir=SUR_DIR)
    R3 = D._read_runs(os.path.join(OUT, "dpc_results.csv"))
    orc = R3[R3.model == "true plant (oracle)"][["seed", "rep", "cost_test_true"]].rename(columns={"cost_test_true": "oracle"})
    R3 = R3.merge(orc, on=["seed", "rep"])
    R3["gap"] = R3.cost_test_true / R3.oracle - 1
    DG = pd.read_csv(os.path.join(OUT, "dpc_diag.csv"))
    DG["rep"] = DG.rep.fillna(0).astype(int)
    R3 = R3.merge(DG[["model", "seed", "rep", "oracle_on_model"]], on=["model", "seed", "rep"], how="left")
    R3["failed"] = R3.gap > 0.10
    R3["trapped"] = R3.failed & (R3.cost_test_own / R3.oracle_on_model > 1.05)
    names = ["LN, MSE (3000)"] + [m["name"] for m in extra] + ["LN, two-stage"]
    T3 = (R3[R3.model.isin(names)].groupby("model")
          .agg(runs=("gap", "size"), failed=("failed", "sum"), trapped=("trapped", "sum"), median_gap_pct=("gap", "median"),
               best_seed_gap_pct=("gap", "min"), s_per_seed=("train_s_per_seed", "median"))
          .reindex(names))
    T3[["median_gap_pct", "best_seed_gap_pct"]] *= 100
    display(T3.round(3))

## Save everything
Download `dpc_out_backup.zip` (Kaggle: Output tab of the committed version). It holds `dpc_results.csv`, `dpc_curves.csv`,
`surrogates.csv`, the tables, figures, trajectories, and the trained surrogates and controllers.

In [ ]:
z = shutil.make_archive(OUT.rstrip("/") + "_backup", "zip", OUT)   # for ARCH != "gru" this zip has no surrogates/ (shared)
print("zipped to", z)
try:
    from google.colab import files
    files.download(z)
except ImportError:
    pass